# Description - To run within SCC
# Creates results using the Residual Pyramid Attention CNN Model
# Adjusted to target models saved by torch_anglepyramidCNN_residual_v3.py
# Path: R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Training_results\momentpyramidCNN_RESIDUAL_v1\

## Cell 1: Imports and Setup

In [1]:
import torch
import json
import torch.nn as nn # <-- ADDED based on residual script
import torch.nn.functional as F # <-- ADDED based on residual script
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import os
from os.path import join
from pickle import load
import pandas as pd
from natsort import natsorted
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
from dataclasses import dataclass, asdict # <-- Keep if needed for metrics later
import math # <-- Keep if needed for metrics later
import subprocess # <-- Keep if needed for xlsxwriter install later
import sys

# Add the path to access helper modules (Windows path)
# KEEPING THIS AS PER INSTRUCTION TO BE EXACT
sys.path.append(r'R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\IMUforKnee\estimation\sensorwise')

# Import custom libraries
# KEEPING THESE AS PER INSTRUCTION TO BE EXACT
try:
    from dir_CBD import *
    from plot_CBD import *
    from scaler_CBD import *
    print("Custom libraries (dir_CBD, plot_CBD, scaler_CBD) imported.")
except ImportError as e:
    print(f"Warning: Could not import custom libraries. Ensure they exist at the specified path. Error: {e}")
    # Define dummy functions if needed later to avoid NameErrors, though they shouldn't be used
    def ensure_dir(directory):
        """Dummy function: Create directory if it doesn't exist."""
        if not os.path.exists(directory):
            os.makedirs(directory)

# Define utility function if not imported (needed later)
if 'ensure_dir' not in globals():
     def ensure_dir(directory):
        """Dummy function: Create directory if it doesn't exist."""
        if not os.path.exists(directory):
            os.makedirs(directory)

Custom libraries (dir_CBD, plot_CBD, scaler_CBD) imported.


## Cell 2: Configuration and Paths

In [2]:
# ==============================================================================
# LOAD TRAINING AND TEST DATA FOR NAIVE BASELINE ANALYSIS - from NPZ
# Loads unscaled ground truth from the original dataset NPZ files
# ==============================================================================
import numpy as np
import os
from os.path import join
from pickle import load # For scaler

def load_ground_truth_for_baseline_from_npz(data_dir, data_type, num_folds=5):
    """
    Load ground truth Y data (train and test) directly from NPZ files and unscale it.
    """
    all_train_true_X, all_train_true_Y, all_train_true_Z = [], [], []
    all_test_true_X, all_test_true_Y, all_test_true_Z = [], [], []

    print("Loading Ground Truth Data for Baseline Analysis from NPZ files...")

    for fold in range(num_folds):
        print(f"\nProcessing Fold {fold}...")
        # Scaler path
        scaler_path = join(data_dir, f"{fold}_fold_scaler4Y_{data_type}.pkl")
        if not os.path.exists(scaler_path):
            print(f"  ❌ Scaler not found: {scaler_path}. Skipping fold.")
            continue
        try:
            # Ensure scaler uses appropriate encoding if needed (e.g., latin1)
            with open(scaler_path, 'rb') as f:
                scaler = load(f) # Add encoding='latin1' if needed
        except Exception as e:
            print(f"  ❌ Error loading scaler {scaler_path}: {e}. Skipping fold.")
            continue


        # --- Load Training Data ---
        train_npz_path = join(data_dir, f"{fold}_fold_final_train.npz")
        if not os.path.exists(train_npz_path):
            print(f"  ❌ Training NPZ not found: {train_npz_path}. Skipping train data for fold.")
        else:
            try:
                train_data = np.load(train_npz_path)
                y_train_key = f"final_Y_{data_type}_train"
                if y_train_key not in train_data:
                    print(f"  ❌ Key '{y_train_key}' not found in {train_npz_path}. Skipping train data.")
                else:
                    y_train_scaled = train_data[y_train_key] # Shape (n_train, 303)
                    n_train_samples = y_train_scaled.shape[0]

                    # Reshape and unscale
                    y_train_reshaped = y_train_scaled.reshape(n_train_samples, 3, 101) # (n, 3, 101)
                    y_train_unscaled = np.zeros_like(y_train_reshaped, dtype=np.float64) # Use float64 for precision

                    # Ensure scaler attributes exist and handle potential shape issues
                    if not hasattr(scaler, 'min_') or not hasattr(scaler, 'scale_'):
                         print(f"  ❌ Scaler object for fold {fold} is missing 'min_' or 'scale_' attributes.")
                         continue
                    expected_scaler_shape = (3,)
                    if scaler.min_.shape != expected_scaler_shape or scaler.scale_.shape != expected_scaler_shape:
                         print(f"  ❌ Scaler shape mismatch for fold {fold}. Expected {expected_scaler_shape}, got min_:{scaler.min_.shape}, scale_:{scaler.scale_.shape}")
                         # Attempt broadcasting if shapes are compatible (e.g., (1,3) vs (3,))
                         try:
                             scaler_min = scaler.min_.reshape(1, 3)
                             scaler_scale = scaler.scale_.reshape(1, 3)
                             print("     Attempting broadcasting for scaler.")
                         except:
                              print("     Cannot reshape scaler for broadcasting. Skipping fold.")
                              continue
                    else:
                        scaler_min = scaler.min_
                        scaler_scale = scaler.scale_


                    for i in range(n_train_samples):
                       # Transpose to (101, 3) for scaler, check shapes before operation
                       sample_T = y_train_reshaped[i].T # Shape (101, 3)
                       if sample_T.shape[1] != scaler_min.shape[-1]:
                           print(f"  ❌ Shape mismatch during unscaling train sample {i}, fold {fold}. Sample shape {sample_T.shape}, scaler min shape {scaler_min.shape}. Skipping sample.")
                           y_train_unscaled[i] = np.nan # Mark as NaN or handle appropriately
                           continue

                       unscaled_sample = (sample_T - scaler_min) / scaler_scale
                       y_train_unscaled[i] = unscaled_sample.T # Back to (3, 101)


                    # Filter out potential NaNs introduced by skipped samples
                    valid_train_mask = ~np.isnan(y_train_unscaled).any(axis=(1, 2))
                    y_train_unscaled_valid = y_train_unscaled[valid_train_mask]

                    if y_train_unscaled_valid.shape[0] > 0:
                        all_train_true_X.extend(y_train_unscaled_valid[:, 0, :]) # Axis 0 (X) - shape (n_valid, 101)
                        all_train_true_Y.extend(y_train_unscaled_valid[:, 1, :]) # Axis 1 (Y)
                        all_train_true_Z.extend(y_train_unscaled_valid[:, 2, :]) # Axis 2 (Z)
                        print(f"  ✅ Loaded and unscaled {y_train_unscaled_valid.shape[0]} valid training trials.")
                    else:
                         print(f"  ⚠️ No valid training trials after unscaling for fold {fold}.")


            except Exception as e:
                print(f"  ❌ Error loading/unscaling training data from {train_npz_path}: {e}")

        # --- Load Test Data ---
        test_npz_path = join(data_dir, f"{fold}_fold_final_test.npz")
        if not os.path.exists(test_npz_path):
             print(f"  ❌ Test NPZ not found: {test_npz_path}. Skipping test data for fold.")
        else:
            try:
                test_data = np.load(test_npz_path)
                y_test_key = f"final_Y_{data_type}_test"
                if y_test_key not in test_data:
                     print(f"  ❌ Key '{y_test_key}' not found in {test_npz_path}. Skipping test data.")
                else:
                    y_test_scaled = test_data[y_test_key] # Shape (n_test, 303)
                    n_test_samples = y_test_scaled.shape[0]

                    # Reshape and unscale (using same scaler checks as train)
                    y_test_reshaped = y_test_scaled.reshape(n_test_samples, 3, 101) # (n, 3, 101)
                    y_test_unscaled = np.zeros_like(y_test_reshaped, dtype=np.float64)
                    if not hasattr(scaler, 'min_') or not hasattr(scaler, 'scale_'): # Redundant check, but safe
                         print(f"  ❌ Scaler object for fold {fold} is missing 'min_' or 'scale_' attributes (test phase).")
                         continue
                    expected_scaler_shape = (3,)
                    if scaler.min_.shape != expected_scaler_shape or scaler.scale_.shape != expected_scaler_shape:
                         print(f"  ❌ Scaler shape mismatch for fold {fold} (test phase). Expected {expected_scaler_shape}, got min_:{scaler.min_.shape}, scale_:{scaler.scale_.shape}")
                         try:
                             scaler_min = scaler.min_.reshape(1, 3)
                             scaler_scale = scaler.scale_.reshape(1, 3)
                             print("     Attempting broadcasting for scaler (test phase).")
                         except:
                              print("     Cannot reshape scaler for broadcasting. Skipping fold (test phase).")
                              continue
                    else:
                        scaler_min = scaler.min_
                        scaler_scale = scaler.scale_

                    for i in range(n_test_samples):
                       sample_T = y_test_reshaped[i].T # Shape (101, 3)
                       if sample_T.shape[1] != scaler_min.shape[-1]:
                           print(f"  ❌ Shape mismatch during unscaling test sample {i}, fold {fold}. Sample shape {sample_T.shape}, scaler min shape {scaler_min.shape}. Skipping sample.")
                           y_test_unscaled[i] = np.nan
                           continue
                       unscaled_sample = (sample_T - scaler_min) / scaler_scale
                       y_test_unscaled[i] = unscaled_sample.T

                    valid_test_mask = ~np.isnan(y_test_unscaled).any(axis=(1, 2))
                    y_test_unscaled_valid = y_test_unscaled[valid_test_mask]

                    if y_test_unscaled_valid.shape[0] > 0:
                        all_test_true_X.extend(y_test_unscaled_valid[:, 0, :])
                        all_test_true_Y.extend(y_test_unscaled_valid[:, 1, :])
                        all_test_true_Z.extend(y_test_unscaled_valid[:, 2, :])
                        print(f"  ✅ Loaded and unscaled {y_test_unscaled_valid.shape[0]} valid test trials.")
                    else:
                         print(f"  ⚠️ No valid test trials after unscaling for fold {fold}.")


            except Exception as e:
                print(f"  ❌ Error loading/unscaling test data from {test_npz_path}: {e}")

    # --- Combine data across folds ---
    train_X = np.array(all_train_true_X) if all_train_true_X else np.empty((0, 101))
    train_Y = np.array(all_train_true_Y) if all_train_true_Y else np.empty((0, 101))
    train_Z = np.array(all_train_true_Z) if all_train_true_Z else np.empty((0, 101))

    test_X = np.array(all_test_true_X) if all_test_true_X else np.empty((0, 101))
    test_Y = np.array(all_test_true_Y) if all_test_true_Y else np.empty((0, 101))
    test_Z = np.array(all_test_true_Z) if all_test_true_Z else np.empty((0, 101))

    print(f"\n✅ Ground truth data loading and unscaling complete!")
    print(f"Training shapes: X={train_X.shape}, Y={train_Y.shape}, Z={train_Z.shape}")
    print(f"Test shapes:     X={test_X.shape}, Y={test_Y.shape}, Z={test_Z.shape}")
    print(f"Total valid training trials loaded: {train_X.shape[0]}, Total valid test trials loaded: {test_X.shape[0]}")

    return train_X, train_Y, train_Z, test_X, test_Y, test_Z

# ==============================================================================
# Configuration for NPZ loading
# ==============================================================================
BASE_DATA_DIR = r"R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Data\allnew_20220325_raw_byDeepak_csv\INC_ByStep\INC_ByZero\Included_checked\SAVE_dataSet"
DATASET_NAME = 'IWALQQ_1st_correction'
DATA_TYPE = 'angle'
DATASET_DIR = join(BASE_DATA_DIR, DATASET_NAME) # This is where the NPZ and PKL files are

# Load the unscaled ground truth data
print("="*80)
print("LOADING UNSCALED GROUND TRUTH DATA FOR BASELINE ANALYSIS (from NPZ)")
print("="*80)
print(f"Looking for NPZ/PKL data in: {DATASET_DIR}")

bl_train_X, bl_train_Y, bl_train_Z, bl_test_X, bl_test_Y, bl_test_Z = load_ground_truth_for_baseline_from_npz(
    data_dir=DATASET_DIR,
    data_type=DATA_TYPE,
    num_folds=5
)

print("\n" + "="*80)
if bl_train_X.shape[0] > 0 and bl_test_X.shape[0] > 0:
    print("✅ ALL GROUND TRUTH DATA READY FOR BASELINE ANALYSIS!")
    print("="*80)
    print("Data summary:")
    print(f"  Training trials: {bl_train_X.shape[0]}")
    print(f"  Test trials: {bl_test_X.shape[0]}")
    if bl_train_X.shape[0] > 0: # Check if training data exists before accessing shape[1]
        print(f"  Time points per axis: {bl_train_X.shape[1]}")
    print("\nYou can now run the naive baseline evaluation.")
else:
     print("❌ FAILED TO LOAD SUFFICIENT DATA FOR BASELINE ANALYSIS!")
     print("   Check the paths and ensure the NPZ/PKL files from the original dataset exist.")
print("="*80)

LOADING UNSCALED GROUND TRUTH DATA FOR BASELINE ANALYSIS (from NPZ)
Looking for NPZ/PKL data in: R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Data\allnew_20220325_raw_byDeepak_csv\INC_ByStep\INC_ByZero\Included_checked\SAVE_dataSet\IWALQQ_1st_correction
Loading Ground Truth Data for Baseline Analysis from NPZ files...

Processing Fold 0...
  ✅ Loaded and unscaled 722 valid training trials.


C:\Users\asmith8\AppData\Local\anaconda3\envs\imu_dl\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  ✅ Loaded and unscaled 154 valid test trials.

Processing Fold 1...
  ✅ Loaded and unscaled 667 valid training trials.
  ✅ Loaded and unscaled 209 valid test trials.

Processing Fold 2...
  ✅ Loaded and unscaled 722 valid training trials.
  ✅ Loaded and unscaled 154 valid test trials.

Processing Fold 3...
  ✅ Loaded and unscaled 670 valid training trials.
  ✅ Loaded and unscaled 206 valid test trials.

Processing Fold 4...
  ✅ Loaded and unscaled 723 valid training trials.
  ✅ Loaded and unscaled 153 valid test trials.

✅ Ground truth data loading and unscaling complete!
Training shapes: X=(3504, 101), Y=(3504, 101), Z=(3504, 101)
Test shapes:     X=(876, 101), Y=(876, 101), Z=(876, 101)
Total valid training trials loaded: 3504, Total valid test trials loaded: 876

✅ ALL GROUND TRUTH DATA READY FOR BASELINE ANALYSIS!
Data summary:
  Training trials: 3504
  Test trials: 876
  Time points per axis: 101

You can now run the naive baseline evaluation.


## Cell 3: Load Ensemble Predictions from NPZ Files

In [ ]:
import numpy as np


def naive_baseline_evaluate(
    train_X, train_Y, train_Z,
    test_X, test_Y, test_Z,
    timepoints_per_axis=101,
    drop_nan_trials=True,
    eps=1e-8
):
    """
    TRUE NAIVE BASELINE: Predicts a single constant value (overall mean across
    ALL training data - all trials, all timepoints) for every timepoint.
    Produces a flat horizontal line prediction.
    """
    axis_labels = ['X', 'Y', 'Z']
    train_arrays = [train_X, train_Y, train_Z]
    test_arrays  = [test_X,  test_Y,  test_Z]

    if train_X.shape[0] == 0 or test_X.shape[0] == 0:
        print("\u274c Cannot run naive baseline: No training or test data loaded.")
        return None

    def _maybe_slice(A, axis_index):
        if A.shape[1] == 3 * timepoints_per_axis:
            sl = slice(axis_index * timepoints_per_axis, (axis_index + 1) * timepoints_per_axis)
            return A[:, sl]
        return A

    results = {}
    print("=" * 80)
    print("TRUE NAIVE BASELINE ANALYSIS (Constant Mean Baseline)")
    print("=" * 80)
    print("Baseline predicts a SINGLE CONSTANT value (overall training mean)")
    print("for every timepoint - flat horizontal line prediction.")
    print("=" * 80)

    for i, ax in enumerate(axis_labels):
        train_axis = _maybe_slice(train_arrays[i], i)
        test_axis  = _maybe_slice(test_arrays[i],  i)

        overall_train_mean = np.nanmean(train_axis)
        train_min  = np.nanmin(train_axis)
        train_max  = np.nanmax(train_axis)
        train_range = max(train_max - train_min, eps)

        baseline_preds = np.full_like(test_axis, overall_train_mean)

        if drop_nan_trials:
            valid_mask = ~np.isnan(test_axis).any(axis=1) & ~np.isnan(baseline_preds).any(axis=1)
        else:
            valid_mask = np.ones(test_axis.shape[0], dtype=bool)

        test_axis_valid = test_axis[valid_mask]
        baseline_valid  = baseline_preds[valid_mask]

        if test_axis_valid.shape[0] == 0:
            results[ax] = {"error": "No valid trials"}
            continue

        flat_true = test_axis_valid.reshape(-1)
        flat_pred = baseline_valid.reshape(-1)
        m = ~np.isnan(flat_true) & ~np.isnan(flat_pred)
        flat_true, flat_pred = flat_true[m], flat_pred[m]

        global_rmse = np.sqrt(np.mean((flat_pred - flat_true) ** 2)) if m.any() else np.nan
        global_corr = (np.corrcoef(flat_true, flat_pred)[0, 1]
                       if flat_true.size > 1 and np.std(flat_true) > eps and np.std(flat_pred) > eps
                       else 0.0)
        global_nrmse_pct = 100 * global_rmse / train_range if not np.isnan(global_rmse) else np.nan

        print(f"\nAxis {ax}")
        print(f"  Constant baseline value: {overall_train_mean:.4f}")
        print(f"  Training range: {train_range:.2f}° (min {train_min:.2f}°, max {train_max:.2f}°)")
        print(f"  Global Corr: {global_corr:.3f} | Global nRMSE: {global_nrmse_pct:.2f}%")

        results[ax] = {
            "global_corr":      global_corr,
            "global_rmse":      global_rmse,
            "global_nrmse_pct": global_nrmse_pct,
            "training_range":   train_range,
        }

    valid_axes = [ax for ax in axis_labels if "error" not in results.get(ax, {})]
    if valid_axes:
        avg_corr  = float(np.nanmean([results[a]["global_corr"]      for a in valid_axes]))
        avg_nrmse = float(np.nanmean([results[a]["global_nrmse_pct"] for a in valid_axes]))
        print("\n" + "=" * 80)
        print("NAIVE BASELINE SUMMARY")
        print("=" * 80)
        print(f"Average global correlation: {avg_corr:.3f}")
        print(f"Average global nRMSE: {avg_nrmse:.2f}%")
        results["summary"] = {"avg_corr": avg_corr, "avg_nrmse_pct": avg_nrmse}

    return results


# --- Run ---
if 'bl_train_X' in locals() and bl_train_X.size > 0:
    naive_results = naive_baseline_evaluate(
        bl_train_X, bl_train_Y, bl_train_Z,
        bl_test_X,  bl_test_Y,  bl_test_Z
    )
else:
    print("\nSkipping Naive Baseline: input data missing.")
    naive_results = None


## Cell 4: Evaluation Metrics Functions

In [4]:
# --- Configuration Section for Residual CNN v3 ---

# Model Version Name (Matches directory structure)
modelVersion = 'anglePyramidCNN_RESIDUAL_v3'

# Dataset details
DATASET_NAME = 'IWALQQ_1st_correction'
DATA_TYPE = 'angle'

# --- Paths based on the Residual CNN script's output structure ---

# Base directory where the training script saved results
BASE_RESULTS_DIR = r'R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Training_results'
RESULTS_DIR = join(BASE_RESULTS_DIR, modelVersion)

# Location of saved model (.pt) files
MODELS_DIR = join(RESULTS_DIR, 'models')

# Location of original data (.npz, .pkl files) - Needed for Inference step
BASE_DATA_DIR = r"R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Data\allnew_20220325_raw_byDeepak_csv\INC_ByStep\INC_ByZero\Included_checked\SAVE_dataSet"
DATASET_DIR = join(BASE_DATA_DIR, DATASET_NAME)

# Base directory for saving output Excel files generated BY THIS NOTEBOOK
# Using the same structure as the original notebook for consistency
outputExcelBaseDir = join(BASE_RESULTS_DIR, modelVersion, DATASET_NAME, DATA_TYPE)
# Ensure this base directory exists for saving later
ensure_dir(outputExcelBaseDir)


# Fixed values
SEQ_LEN = 101
NUM_FEATURES = 42

# AI training settings (Device needed for inference)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("RESIDUAL CNN MODELS_DIR:", MODELS_DIR)
print("DATA_DIR (for inference input):", DATASET_DIR)
print("Output Excel Base Dir (this notebook):", outputExcelBaseDir)


# Quick check of available model files
import os
if os.path.exists(MODELS_DIR):
    print("\nRESIDUAL CNN Model files found:")
    model_files = sorted([f for f in os.listdir(MODELS_DIR) if f.endswith('.pt')])
    if model_files:
        for file in model_files:
            print(f"  - {file}")
    else:
        print("  No .pt model files found in the directory.")
else:
    print(f"\n❌ ERROR: RESIDUAL CNN model directory not found: {MODELS_DIR}")
    print("      Ensure the training script ran and saved models here.")

# Check for data files needed for inference
if not os.path.exists(DATASET_DIR):
     print(f"\n❌ ERROR: Original data directory not found: {DATASET_DIR}")
     print("      NPZ and PKL files needed for inference are missing.")

Device: cuda
RESIDUAL CNN MODELS_DIR: R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Training_results\anglePyramidCNN_RESIDUAL_v3\models
DATA_DIR (for inference input): R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Data\allnew_20220325_raw_byDeepak_csv\INC_ByStep\INC_ByZero\Included_checked\SAVE_dataSet\IWALQQ_1st_correction
Output Excel Base Dir (this notebook): R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Training_results\anglePyramidCNN_RESIDUAL_v3\IWALQQ_1st_correction\angle

RESIDUAL CNN Model files found:
  - fold_0_ensemble_models.pt
  - fold_0_seed_123_model.pt
  - fold_0_seed_42_model.pt
  - fold_0_seed_456_model.pt
  - fold_1_ensemble_models.pt
  - fold_1_seed_123_model.pt
  - fold_1_seed_42_model.pt
  - fold_1_seed_456_model.pt
  - fold_2_ensemble_models.pt
  - fold_2_seed_123_model.pt
  - fold_2_seed_42_model.pt
  - fold_2_seed_456_model.pt
  - fold_3_ensemble_models.pt
  - fold_3_seed_123_model.pt
  - 

## Cell 5: Evaluate Training Set

In [5]:
# --- Residual CNN v3 MODEL CLASSES ---
# Required for loading the saved models during inference
import torch
import torch.nn as nn
import torch.nn.functional as F

class SafeBatchNorm1d(nn.Module):
    """BatchNorm1d that handles single-sample batches safely for Conv layers."""
    def __init__(self, num_features, **kwargs):
        super().__init__()
        self.bn = nn.BatchNorm1d(num_features, **kwargs)

    def forward(self, x):
        # Original script checks self.training, keep that for consistency if loading state_dict expects it
        # If loading only for eval, this check might not be strictly necessary, but safer to keep.
        if self.training and x.size(0) == 1:
            return x
        # During eval (or batch size > 1 during training), use BatchNorm
        # Check for non-finite values before BatchNorm during evaluation
        if not self.training:
             if not torch.all(torch.isfinite(x)):
                  print("Warning: Non-finite values detected before BatchNorm1d during evaluation.")
                  # Option 1: Skip BN (might be okay if variance is stable)
                  # return x
                  # Option 2: Replace NaNs/Infs (safer but might mask issues)
                  x = torch.nan_to_num(x, nan=0.0, posinf=1e6, neginf=-1e6) # Example values
        try:
             out = self.bn(x)
             # Check for non-finite values *after* BatchNorm
             if not torch.all(torch.isfinite(out)):
                  print("Warning: Non-finite values detected *after* BatchNorm1d during evaluation.")
                  # Option: return input x, or handle as needed
                  # return x # Fallback to input
                  return torch.nan_to_num(out, nan=0.0, posinf=1e6, neginf=-1e6)
             return out
        except ValueError as e:
             print(f"BatchNorm1d error during evaluation: {e}. Input shape: {x.shape}. Returning input.")
             return x # Fallback to input if BN fails

class ResidualBlock(nn.Module):
    """
    Residual block with skip connection for 1D convolutions.
    Implements: output = F(x) + x, where F(x) is Conv->BN->ReLU->Dropout
    Uses 1x1 convolution for skip connection when channels don't match.
    """
    def __init__(self, in_channels, out_channels, kernel_size, dropout=0.0):
        super().__init__()
        # Main path: Conv -> BN -> ReLU -> Dropout
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size,
                              padding=kernel_size // 2)
        self.bn = SafeBatchNorm1d(out_channels)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # Skip connection: use 1x1 conv if channels don't match, else identity
        if in_channels != out_channels:
            self.skip = nn.Conv1d(in_channels, out_channels, kernel_size=1)
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        # Main path
        out = self.conv(x)
        out = self.bn(out)
        out = self.relu(out)
        out = self.dropout(out)

        # Add skip connection
        skip = self.skip(x)
        out = out + skip
        return out

class PyramidAttnCNN(nn.Module):
    """
    V3: Multi-scale CNN with INITIAL FEATURE EXPANSION, residual connections, and attention.
    
    Architecture:
    1. Initial expansion layer: 42 -> 64 channels (1x1 conv + BN + ReLU)
    2. Multi-branch pyramid with residual blocks
    3. Attention mechanism
    4. Output heads
    """
    def __init__(self, kernels, channels, n_layers,
                 dropout_conv, dropout_fc, num_heads=4,
                 num_features=42, num_outputs=303,
                 initial_expansion=64):
        super().__init__()
        
        # V3: Initial feature expansion layer (42 -> 64 channels)
        self.initial_expansion = nn.Sequential(
            nn.Conv1d(num_features, initial_expansion, kernel_size=1),
            SafeBatchNorm1d(initial_expansion),
            nn.ReLU()
        )
        
        self.branches = nn.ModuleList()

        for k in kernels:
            layers = []
            in_ch = initial_expansion  # V3: Start from expanded features, not original 42
            # Build residual blocks for each layer
            for i in range(n_layers):
                out_ch = channels[min(i, len(channels) - 1)]
                layers.append(ResidualBlock(in_ch, out_ch, kernel_size=k,
                                           dropout=dropout_conv))
                in_ch = out_ch

            # Keep strided convolutions unchanged (temporal pyramid)
            layers.append(nn.Conv1d(out_ch, out_ch, kernel_size=3, stride=4, padding=1))
            layers.append(SafeBatchNorm1d(out_ch))
            layers.append(nn.ReLU())
            layers.append(nn.Conv1d(out_ch, out_ch, kernel_size=3, stride=3, padding=1))
            layers.append(SafeBatchNorm1d(out_ch))
            layers.append(nn.ReLU())
            self.branches.append(nn.Sequential(*layers))

        embed_dim = channels[min(n_layers - 1, len(channels) - 1)] * len(kernels)
        # Ensure embed_dim is divisible by 2 for the reduction layer
        if embed_dim % 2 != 0:
             # Handle odd embed_dim, maybe by adjusting the last channel count or padding
             # For simplicity here, let's adjust reduction layer output, though architecture might differ
             print(f"Warning: embed_dim ({embed_dim}) is odd. Adjusting reduction layer output.")
             reduced_dim = (embed_dim // 2) + 1 # Or handle differently based on exact trained model
        else:
             reduced_dim = embed_dim // 2

        self.reduce = nn.Sequential(
            nn.Conv1d(embed_dim, reduced_dim, kernel_size=1),
            SafeBatchNorm1d(reduced_dim),
            nn.ReLU(),
        )
        attn_dim = reduced_dim
        # Ensure attn_dim is divisible by num_heads
        if attn_dim % num_heads != 0:
            print(f"Warning: attn_dim ({attn_dim}) is not divisible by num_heads ({num_heads}). Adjusting num_heads.")
            # Find a divisor of attn_dim, e.g., 2 or 1 if necessary
            num_heads = 1
            for h in range(min(4, attn_dim), 0, -1): # Check 4, 3, 2, 1
                 if attn_dim % h == 0:
                      num_heads = h
                      break
            print(f"  Using adjusted num_heads = {num_heads}")

        self.attn = nn.MultiheadAttention(attn_dim, num_heads=num_heads, batch_first=True, dropout=0.1) # Added dropout based on script
        # self.dropout_attn = nn.Dropout(0.1) # Dropout included in MultiheadAttention

        self.heads = nn.ModuleDict({
            'X': nn.Linear(attn_dim, 101),
            'Y': nn.Linear(attn_dim, 101),
            'Z': nn.Linear(attn_dim, 101),
        })
        # Dropout_fc might have been intended for a layer after attention/pooling,
        # but isn't explicitly used here in the script's final layer definition.
        # Adding a dropout layer before the final heads if needed:
        self.dropout_final = nn.Dropout(dropout_fc) if dropout_fc > 0 else nn.Identity()


    def forward(self, x):
        # Input shape check
        if x.shape[1] != 42 or x.shape[2] != 101:
            print(f"Warning: Unexpected input shape to PyramidAttnCNN: {x.shape}. Expected (N, 42, 101).")
            # Attempt to reshape if possible, or raise error
            # Example: Reshape if flattened input (N, 42*101) was given
            # if x.ndim == 2 and x.shape[1] == 42 * 101:
            #     x = x.view(-1, 42, 101)
            # else:
            #     raise ValueError("Cannot process input shape.")

        # V3: First expand features from 42 -> 64
        x = self.initial_expansion(x)
        
        # Then process through pyramid branches
        branch_feats = [br(x) for br in self.branches]
        try:
             x = torch.cat(branch_feats, dim=1)
        except RuntimeError as e:
             print(f"Error concatenating branch features: {e}")
             print("Branch output shapes:", [f.shape for f in branch_feats])
             raise e

        x = self.reduce(x)
        seq = x.permute(0, 2, 1) # Shape (N, SeqReduced, attn_dim)
        try:
             # Ensure seq is not empty or has zero dimension
             if seq.nelement() == 0 or seq.shape[1] == 0:
                  print("Warning: Sequence input to attention is empty or has zero length.")
                  # Handle this case, e.g., return zeros or raise error
                  attn_dim = self.heads['X'].in_features
                  zero_output = torch.zeros(x.shape[0], 303, device=x.device)
                  return zero_output

             seq, _ = self.attn(seq, seq, seq) # Apply attention
             # seq = self.dropout_attn(seq) # Dropout is now within MultiheadAttention
        except Exception as e:
            print(f"Error during MultiheadAttention: {e}")
            print("Input sequence shape:", seq.shape)
            raise e

        pooled = seq.mean(dim=1) # Global average pooling over sequence dimension
        pooled = self.dropout_final(pooled) # Apply final dropout if defined

        out_x = self.heads['X'](pooled)
        out_y = self.heads['Y'](pooled)
        out_z = self.heads['Z'](pooled)
        return torch.cat([out_x, out_y, out_z], dim=1)

print("Residual CNN v3 model classes defined.")

Residual CNN v3 model classes defined.


## Cell 6: Evaluate Test Set

In [6]:
# --- ENSEMBLE INFERENCE CODE - Adapted for Residual CNN v2 ---
from os.path import join
import os
import numpy as np
import pandas as pd
import torch
from pickle import load

print("\n" + "="*80)
print("PERFORMING INFERENCE ON TEST SET using Residual CNN v2 Ensemble")
print("="*80)

# Make sure these are correctly defined from Cell 5
# MODELS_DIR: Directory containing the trained .pt files
# DATASET_DIR: Directory containing the .npz data files and .pkl scaler files
# outputExcelBaseDir: Base directory to save the output Excel files

# Load the hyperparameters used for training from the config file saved by the script
# Find the config file (assuming one exists)
config_files = [f for f in os.listdir(RESULTS_DIR) if f.startswith('ensemble_config_') and f.endswith('.json')]
if not config_files:
    print("❌ ERROR: Cannot find ensemble_config_*.json file in RESULTS_DIR.")
    print("   Cannot determine the exact hyperparameters used for training.")
    # Fallback to BEST_PARAMS defined earlier, but warn user
    print("   Warning: Falling back to potentially outdated BEST_PARAMS dictionary.")
    BEST_PARAMS_USED = BEST_PARAMS # Defined in the original notebook cell 5 copy
else:
    # Load the latest config file
    config_file_path = join(RESULTS_DIR, sorted(config_files)[-1])
    try:
        with open(config_file_path, 'r') as f:
            training_config = json.load(f)
        BEST_PARAMS_USED = training_config['best_params']
        print(f"Loaded hyperparameters from: {config_file_path}")
    except Exception as e:
        print(f"❌ ERROR: Could not load or parse config file {config_file_path}: {e}")
        print("   Warning: Falling back to potentially outdated BEST_PARAMS dictionary.")
        BEST_PARAMS_USED = BEST_PARAMS

# Ensure required keys are present
required_keys = ['kernels', 'channels', 'n_layers', 'dropout_conv', 'dropout_fc', 'num_heads']
if not all(key in BEST_PARAMS_USED for key in required_keys):
     print("❌ ERROR: Loaded config is missing required keys for model instantiation.")
     print(f"   Required: {required_keys}")
     print(f"   Found in config: {list(BEST_PARAMS_USED.keys())}")
     # Handle error appropriately, e.g., stop execution or use defaults with warning
     raise KeyError("Missing critical hyperparameters in loaded config.")


total_files_processed = 0
for numFold in range(5):
    print(f"\n--- Processing Fold {numFold} ---")
    # 1. Load ensemble checkpoint (containing model states and config)
    ensemble_path = join(MODELS_DIR, f'fold_{numFold}_ensemble_models.pt')
    if not os.path.exists(ensemble_path):
        print(f"  Ensemble file not found: {ensemble_path}. Skipping fold.")
        continue

    try:
        # Load checkpoint with appropriate map_location
        ckpt = torch.load(ensemble_path, map_location=torch.device(device)) # Load to target device
        state_dicts = ckpt.get('models')
        # Use config saved WITHIN the checkpoint if available, otherwise fallback
        ensemble_config = ckpt.get('config', BEST_PARAMS_USED)
        model_seeds = ckpt.get('seeds', ['unknown'] * len(state_dicts) if state_dicts else []) # Get seeds if available
        if state_dicts is None:
             print(f"  ❌ Key 'models' not found in checkpoint: {ensemble_path}. Skipping fold.")
             continue
        print(f"  Fold {numFold}: loaded ensemble with {len(state_dicts)} models (Seeds: {model_seeds}).")

    except Exception as e:
        print(f"  ❌ Error loading checkpoint {ensemble_path}: {e}. Skipping fold.")
        continue

    # 2. Extract PyramidAttnCNN init args from the loaded config
    # Ensure all necessary keys are present in ensemble_config
    try:
        cnn_kwargs = {
            'kernels':      ensemble_config['kernels'],
            'channels':     ensemble_config['channels'],
            'n_layers':     ensemble_config['n_layers'],
            'dropout_conv': ensemble_config['dropout_conv'],
            'dropout_fc':   ensemble_config['dropout_fc'],
            'num_heads':    ensemble_config.get('num_heads', 4), # Use default if missing
            # Use defaults for num_features=42 and num_outputs=303
        }
    except KeyError as e:
        print(f"  ❌ Missing key '{e}' in loaded ensemble config for fold {numFold}. Using default BEST_PARAMS.")
        # Fallback to the globally defined BEST_PARAMS if checkpoint config is bad
        try:
            cnn_kwargs = {
                'kernels':      BEST_PARAMS['kernels'],
                'channels':     BEST_PARAMS['channels'],
                'n_layers':     BEST_PARAMS['n_layers'],
                'dropout_conv': BEST_PARAMS['dropout_conv'],
                'dropout_fc':   BEST_PARAMS['dropout_fc'],
                'num_heads':    BEST_PARAMS.get('num_heads', 4),
            }
        except KeyError as e_fallback:
             print(f"  ❌ Missing key '{e_fallback}' even in fallback BEST_PARAMS. Cannot build model. Skipping fold.")
             continue


    # 3. Rebuild each ensemble member using PyramidAttnCNN
    ensemble_models = []
    models_loaded_successfully = True
    for idx, sd in enumerate(state_dicts):
        try:
            m = PyramidAttnCNN(**cnn_kwargs)
            m.load_state_dict(sd)
            m.to(device) # Move model to device
            m.eval() # Set to evaluation mode
            ensemble_models.append(m)
        except Exception as e:
            print(f"  ❌ Error loading state dict for model {idx} in fold {numFold}: {e}")
            models_loaded_successfully = False
            break # Stop processing this fold if a model fails to load

    if not models_loaded_successfully or not ensemble_models:
        print(f"  Skipping inference for fold {numFold} due to model loading errors.")
        continue

    # 4. Load test data features + output scaler
    load_test_path = join(DATASET_DIR, f"{numFold}_fold_final_test.npz")
    scaler_path = join(DATASET_DIR, f"{numFold}_fold_scaler4Y_{DATA_TYPE}.pkl")

    if not os.path.exists(load_test_path) or not os.path.exists(scaler_path):
        print(f"  ❌ Test NPZ or Scaler PKL file missing for fold {numFold}. Skipping.")
        continue

    try:
        load_test_data = np.load(load_test_path)
        # Check if keys exist
        x_test_key = "final_X_test"
        y_test_key = f"final_Y_{DATA_TYPE}_test"
        if x_test_key not in load_test_data or y_test_key not in load_test_data:
             print(f"  ❌ Keys '{x_test_key}' or '{y_test_key}' not found in {load_test_path}. Skipping fold.")
             continue
        X_test_all = load_test_data[x_test_key]
        Y_true_scaled_all = load_test_data[y_test_key]

        with open(scaler_path, "rb") as f:
             load_scaler4Y = load(f) # Add encoding='latin1' if needed

        n_samples = X_test_all.shape[0]
        print(f"  Processing fold {numFold} with {n_samples} test samples")

    except Exception as e:
        print(f"  ❌ Error loading test data or scaler for fold {numFold}: {e}. Skipping.")
        continue


    # 5. Inference loop: average across ensemble, reshape, unscale, save
    fold_files_processed = 0
    for dataIndex in range(n_samples):
        # Input features: Reshape (42*101) -> (1, 42, 101) and convert to tensor
        X_sample = X_test_all[dataIndex].reshape(1, NUM_FEATURES, SEQ_LEN) # Add batch dim
        Y_true_scaled_sample = Y_true_scaled_all[dataIndex] # Shape (303,)

        # Convert X to tensor and move to device
        x_tensor = torch.from_numpy(X_sample).float().to(device)

        with torch.no_grad():
            preds_scaled = []
            for model in ensemble_models:
                # Ensure model and data are on the same device
                pred_scaled = model(x_tensor)
                preds_scaled.append(pred_scaled.cpu().numpy()) # Move back to CPU for numpy

            # Average predictions across the ensemble
            Y_pred_scaled_avg = np.mean(np.stack(preds_scaled, axis=0), axis=0) # Shape (1, 303)

        # Reshape predicted and true (scaled) to (101, 3) for unscaling
        Y_pred_scaled_reshaped = Y_pred_scaled_avg.reshape(3, -1).T # (101, 3)
        Y_true_scaled_reshaped = Y_true_scaled_sample.reshape(3, -1).T # (101, 3)

        # Un-scale using the loaded scaler
        try:
            # Check scaler attribute shapes again just before use
            if load_scaler4Y.min_.shape != (3,) or load_scaler4Y.scale_.shape != (3,):
                 print(f"    ❌ Scaler shape mismatch for fold {numFold} during unscaling. Skipping sample {dataIndex}.")
                 continue

            Y_true_unscaled = (Y_true_scaled_reshaped - load_scaler4Y.min_) / load_scaler4Y.scale_
            Y_pred_unscaled = (Y_pred_scaled_reshaped - load_scaler4Y.min_) / load_scaler4Y.scale_
        except Exception as e:
            print(f"    ❌ Error unscaling data for sample {dataIndex}, fold {numFold}: {e}. Skipping sample.")
            continue


        # Save unscaled results to Excel
        df_true = pd.DataFrame(Y_true_unscaled, columns=['X_True','Y_True','Z_True'])
        df_pred = pd.DataFrame(Y_pred_unscaled, columns=['X_Pred','Y_Pred','Z_Pred'])
        df_save = pd.concat([df_true, df_pred], axis=1)

        # Define output directory for this fold's results
        out_dir_fold = join(outputExcelBaseDir, f"{numFold}_fold")
        ensure_dir(out_dir_fold) # Make sure directory exists

        excel_filename = f"{dataIndex}.xlsx"
        excel_filepath = join(out_dir_fold, excel_filename)
        try:
            df_save.to_excel(excel_filepath, index=True, index_label="Timepoint") # Save timepoint index
            fold_files_processed += 1
        except Exception as e:
            print(f"    ❌ Error saving Excel file {excel_filepath}: {e}")

    print(f"  Fold {numFold} completed successfully! Processed {fold_files_processed}/{n_samples} samples.")
    total_files_processed += fold_files_processed

print("\n" + "="*80)
print(f"All Residual CNN v2 TEST folds processed. Total files saved: {total_files_processed}")
print("="*80)


PERFORMING INFERENCE ON TEST SET using Residual CNN v2 Ensemble
Loaded hyperparameters from: R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Training_results\anglePyramidCNN_RESIDUAL_v3\ensemble_config_20260303_084634.json

--- Processing Fold 0 ---


C:\Users\asmith8\AppData\Local\Temp\ipykernel_3168\2607697363.py:61: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ensemble_path, map_location=torch.device

  Fold 0: loaded ensemble with 3 models (Seeds: [42, 123, 456]).
  Processing fold 0 with 154 test samples
  Fold 0 completed successfully! Processed 154/154 samples.

--- Processing Fold 1 ---


C:\Users\asmith8\AppData\Local\Temp\ipykernel_3168\2607697363.py:61: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ensemble_path, map_location=torch.device

  Fold 1: loaded ensemble with 3 models (Seeds: [42, 123, 456]).


...

  Processing fold 3 with 206 test samples
  Fold 3 completed successfully! Processed 206/206 samples.

--- Processing Fold 4 ---


C:\Users\asmith8\AppData\Local\Temp\ipykernel_3168\2607697363.py:61: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ensemble_path, map_location=torch.device

  Fold 4: loaded ensemble with 3 models (Seeds: [42, 123, 456]).


C:\Users\asmith8\AppData\Local\anaconda3\envs\imu_dl\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  Processing fold 4 with 153 test samples
  Fold 4 completed successfully! Processed 153/153 samples.

All Residual CNN v2 TEST folds processed. Total files saved: 876


## Cell 7: Generalization Analysis (Train vs Test Gap)

In [7]:
# import 필요한 라이브러리 (Some might be redundant)
import os
from os.path import join
from natsort import natsorted
from pathlib import Path
import shutil
import pandas as pd
import numpy as np
# from tqdm.notebook import tqdm # Not used in original notebook's subsequent cells
from mpl_toolkits.axes_grid1 import host_subplot # Not used in original plots
import mpl_toolkits.axisartist as AA # Not used in original plots
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
# import plot_CBD # Custom library - not needed if replicating plots directly

## Cell 8: Save Comprehensive Results to JSON

In [8]:
# Define the target directory where the inference step (Cell 7) saved the Excel outputs
# This MUST match the 'outputExcelBaseDir' used in Cell 7
outputExcelBaseDir = outputExcelBaseDir # Already defined in Cell 5
print(f"Target directory for reading aggregated results: {outputExcelBaseDir}")

Target directory for reading aggregated results: R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Training_results\anglePyramidCNN_RESIDUAL_v3\IWALQQ_1st_correction\angle


## Cell 9: Export Predictions to Excel (Optional)

In [9]:
# --- Aggregate TEST Set Results from Excel Files ---
print("\nAggregating TEST set results from Excel files...")
# Initialize empty numpy arrays
true_X = np.array([])
pred_X = np.array([])
true_Y = np.array([])
pred_Y = np.array([])
true_Z = np.array([])
pred_Z = np.array([])

test_folders = [f"{fold}_fold" for fold in range(5)]
total_test_files_loaded_agg = 0
problematic_files = []

for fold in test_folders:
    fold_path = join(outputExcelBaseDir, fold)
    if not os.path.exists(fold_path):
        print(f"  Skipping non-existent test fold for aggregation: {fold}")
        continue
    print(f"  Aggregating from test fold: {fold}")
    list_results = natsorted([f for f in os.listdir(fold_path) if f.endswith(".xlsx")])
    fold_files_loaded = 0
    for result_file in list_results:
        file_path = join(fold_path, result_file)
        try:
            # Use index_col=0 if "Timepoint" was saved as index, otherwise default is None
            singleResult = pd.read_excel(file_path, index_col=0) # Adjust if index wasn't saved

            # Check if DataFrame has expected 101 rows (timepoints)
            if singleResult.shape[0] != 101:
                print(f"    Warning: File {result_file} does not have 101 rows (shape: {singleResult.shape}). Skipping.")
                problematic_files.append(file_path)
                continue

            loaded_axes_in_file = 0
            temp_true = {}
            temp_pred = {}
            valid_file = True
            for axe in ["X", "Y", "Z"]:
                true_col = f"{axe}_True"
                pred_col = f"{axe}_Pred"
                if true_col in singleResult.columns and pred_col in singleResult.columns:
                    # Extract data, check for NaNs, and reshape
                    true_vals = singleResult[true_col].to_numpy()
                    pred_vals = singleResult[pred_col].to_numpy()

                    if np.isnan(true_vals).any() or np.isnan(pred_vals).any():
                         print(f"    Warning: NaNs found in axis {axe} of {result_file}. Skipping file.")
                         problematic_files.append(file_path + f" (NaNs in axis {axe})")
                         valid_file = False
                         break # Skip rest of axes for this file

                    temp_true[axe] = np.expand_dims(true_vals, axis=0) # Shape (1, 101)
                    temp_pred[axe] = np.expand_dims(pred_vals, axis=0) # Shape (1, 101)
                    loaded_axes_in_file += 1
                else:
                    print(f"    Warning: Missing columns for axis {axe} in {result_file}. Skipping file.")
                    problematic_files.append(file_path + f" (Missing cols for axis {axe})")
                    valid_file = False
                    break # Skip rest of axes

            if valid_file and loaded_axes_in_file == 3:
                # Append data for all axes if file was valid
                for axe in ["X", "Y", "Z"]:
                    current_true = globals().get(f"true_{axe}")
                    current_pred = globals().get(f"pred_{axe}")
                    if current_true is not None and current_true.size == 0:
                        globals()[f"true_{axe}"] = temp_true[axe]
                        globals()[f"pred_{axe}"] = temp_pred[axe]
                    elif current_true is not None:
                        globals()[f"true_{axe}"] = np.concatenate((current_true, temp_true[axe]), 0)
                        globals()[f"pred_{axe}"] = np.concatenate((current_pred, temp_pred[axe]), 0)
                fold_files_loaded += 1
            elif valid_file and loaded_axes_in_file != 3:
                 print(f"    Warning: File {result_file} loaded successfully but didn't contain all 3 axes. Skipping aggregation for this file.")
                 problematic_files.append(file_path + f" (Incomplete axes)")


        except Exception as e:
            print(f"    ❌ Error aggregating file {result_file}: {e}")
            problematic_files.append(file_path + f" (Read Error: {e})")

    print(f"    Aggregated {fold_files_loaded} valid files from {fold}")
    total_test_files_loaded_agg += fold_files_loaded

print(f"\nTotal TEST files aggregated: {total_test_files_loaded_agg}")
print("Aggregated test array shapes:")
print(f"  True X: {true_X.shape}, Pred X: {pred_X.shape}")
print(f"  True Y: {true_Y.shape}, Pred Y: {pred_Y.shape}")
print(f"  True Z: {true_Z.shape}, Pred Z: {pred_Z.shape}")

if problematic_files:
     print("\nProblematic files encountered during aggregation:")
     for f in problematic_files[:10]: # Print first 10 issues
          print(f"  - {f}")
     if len(problematic_files) > 10:
          print(f"  ... and {len(problematic_files)-10} more.")

print("\n✅ Test set aggregation complete.")


Aggregating TEST set results from Excel files...
  Aggregating from test fold: 0_fold
    Aggregated 154 valid files from 0_fold
  Aggregating from test fold: 1_fold
    Aggregated 209 valid files from 1_fold
  Aggregating from test fold: 2_fold
    Aggregated 154 valid files from 2_fold
  Aggregating from test fold: 3_fold
    Aggregated 206 valid files from 3_fold
  Aggregating from test fold: 4_fold
    Aggregated 153 valid files from 4_fold

Total TEST files aggregated: 876
Aggregated test array shapes:
  True X: (876, 101), Pred X: (876, 101)
  True Y: (876, 101), Pred Y: (876, 101)
  True Z: (876, 101), Pred Z: (876, 101)

✅ Test set aggregation complete.


## Cell 10: Visualization - Prediction Scatter Plots

In [10]:
def makeDataframe(file_Num):
    # Ensure input is numpy array
    if isinstance(file_Num, pd.DataFrame):
        file_Num = file_Num.to_numpy()
    # Check if array is empty
    if file_Num.size == 0:
        print("Warning: makeDataframe received an empty array.")
        return pd.DataFrame() # Return empty DataFrame
    # Transpose: from (trials, time) to (time, trials)
    return pd.DataFrame(data=file_Num.transpose(),
                        index = [idx for idx in range(file_Num.shape[1])], # time indices
                        columns=[idx for idx in range(file_Num.shape[0])]) # trial indices

## Cell 11: Visualization - Error Distribution Plots

In [11]:
# --- Rebuild arrays from the aggregated Excel files (Redundant step, kept for structural similarity) ---
# This part is technically redundant if Cell 10 worked correctly and variables are in memory.
# It assumes Cell 13 successfully created 'TruePredDiff.xlsx'.
print("\nRe-reading aggregated TEST data from TruePredDiff.xlsx (structural step)...")

test_excel_path = join(outputExcelBaseDir, 'TruePredDiff.xlsx')
if os.path.exists(test_excel_path):
    true_X_read = np.array([])
    pred_X_read = np.array([])
    true_Y_read = np.array([])
    pred_Y_read = np.array([])
    true_Z_read = np.array([])
    pred_Z_read = np.array([])
    read_successful = True
    try:
        xls = pd.ExcelFile(test_excel_path)
        for axe in ["X", "Y", "Z"]:
            if f"true_{axe}" in xls.sheet_names:
                # Read, transpose back to (trials, time)
                df_true = pd.read_excel(xls, sheet_name=f"true_{axe}", index_col=0)
                globals()[f"true_{axe}_read"] = df_true.transpose().to_numpy()
            else:
                 print(f"  Warning: Sheet 'true_{axe}' not found in {test_excel_path}")
                 read_successful = False

            if f"pred_{axe}" in xls.sheet_names:
                df_pred = pd.read_excel(xls, sheet_name=f"pred_{axe}", index_col=0)
                globals()[f"pred_{axe}_read"] = df_pred.transpose().to_numpy()
            else:
                 print(f"  Warning: Sheet 'pred_{axe}' not found in {test_excel_path}")
                 read_successful = False
        if read_successful:
             # If read was successful, optionally overwrite original variables
             # true_X, pred_X = true_X_read, pred_X_read
             # true_Y, pred_Y = true_Y_read, pred_Y_read
             # true_Z, pred_Z = true_Z_read, pred_Z_read
             print("  Successfully re-read data from Test Excel file.")
             print("  Test array shapes (re-read):")
             print(f"    True X: {true_X_read.shape}, Pred X: {pred_X_read.shape}")
             print(f"    True Y: {true_Y_read.shape}, Pred Y: {pred_Y_read.shape}")
             print(f"    True Z: {true_Z_read.shape}, Pred Z: {pred_Z_read.shape}")
        else:
             print("  Could not re-read all required sheets from Test Excel file. Using arrays from previous aggregation.")

    except Exception as e:
        print(f"  ❌ Error re-reading {test_excel_path}: {e}. Using arrays from previous aggregation.")

else:
    print(f"  {test_excel_path} not found. Using arrays from previous aggregation.")

print("Test arrays ready for saving.") # Keep original print statement intent


Re-reading aggregated TEST data from TruePredDiff.xlsx (structural step)...
  R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Training_results\anglePyramidCNN_RESIDUAL_v3\IWALQQ_1st_correction\angle\TruePredDiff.xlsx not found. Using arrays from previous aggregation.
Test arrays ready for saving.


In [12]:
# --- Save Aggregated TEST Data to Multi-Sheet Excel ---
# Install xlsxwriter if not available
try:
    import xlsxwriter
except ImportError:
    import subprocess
    import sys
    print("Installing xlsxwriter...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xlsxwriter"])
    import xlsxwriter
    print("xlsxwriter installed.")

test_excel_path_save = join(outputExcelBaseDir,'TruePredDiff.xlsx') # Path defined in previous cell
print(f"\nSaving aggregated TEST results to: {test_excel_path_save}")

# Check if data exists before writing
if 'true_X' not in locals() or true_X.size == 0:
     print("  No aggregated TEST data available to save. Skipping Excel write.")
else:
    try:
        writer_save_test = pd.ExcelWriter(test_excel_path_save, engine='xlsxwriter')
        write_success = True
        for axe in ["X","Y","Z"]:
            for sess in ["true","pred"]:
                dataName = f"{sess}_{axe}"
                if dataName in globals() and globals()[dataName].size > 0:
                    df_to_save = makeDataframe(globals()[dataName])
                    # Ensure DataFrame is not empty before saving
                    if not df_to_save.empty:
                         df_to_save.to_excel(writer_save_test, sheet_name=str(dataName), index=True, index_label="Timepoint") # Save index
                    else:
                         print(f"  Skipping empty sheet: {dataName}")
                else:
                    print(f"  Warning: Data array '{dataName}' not found or empty. Skipping sheet.")
                    write_success = False # Mark as potentially incomplete

            # Calculate and save differences
            true_data_save = globals().get(f"true_{axe}")
            pred_data_save = globals().get(f"pred_{axe}")
            if true_data_save is not None and pred_data_save is not None and \
               true_data_save.size > 0 and pred_data_save.size > 0 and \
               true_data_save.shape == pred_data_save.shape:
                df_diff_save = makeDataframe(abs(true_data_save - pred_data_save))
                if not df_diff_save.empty:
                     df_diff_save.to_excel(writer_save_test, sheet_name=str(f"diff_{axe}"), index=True, index_label="Timepoint")
                else:
                     print(f"  Skipping empty diff sheet: diff_{axe}")
            else:
                 print(f"  Warning: Cannot calculate or save diff for axis {axe} due to missing/mismatched data.")
                 write_success = False

        writer_save_test.close()
        if write_success:
             print("  Test data Excel file saved successfully.")
        else:
             print("  Test data Excel file saved, but some sheets might be missing.")

    except Exception as e:
        print(f"  ❌ Error writing TEST Excel file {test_excel_path_save}: {e}")


Saving aggregated TEST results to: R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Training_results\anglePyramidCNN_RESIDUAL_v3\IWALQQ_1st_correction\angle\TruePredDiff.xlsx
  Test data Excel file saved successfully.


In [ ]:
# --- Generate TEST Set Summary Plot ---
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import matplotlib.pyplot as plt

# Check if aggregated data exists
if 'true_X' not in locals() or true_X.size == 0:
    print("\nSkipping TEST set summary plot generation (no aggregated data).")
else:
    pdf_path_test_plot = join(outputExcelBaseDir, f"WHOLE_Total_result_RESIDUAL_v3_TEST_{DATA_TYPE}.pdf")
    print(f"\nGenerating TEST set summary plot: {pdf_path_test_plot}")
    try:
        pp_test_plot = PdfPages(pdf_path_test_plot)
        fig_test_plot, axes_test_plot = plt.subplots(3, 2, figsize=(12, 8), sharex=True)
        time_plot = np.linspace(0, 100, 101) # X-axis represents % of cycle or normalized time
        angle_labels_plot = ['X', 'Y', 'Z']

        plot_successful = True
        for i, (T_plot, P_plot) in enumerate(zip([true_X, true_Y, true_Z], [pred_X, pred_Y, pred_Z])):
           # Double-check arrays before plotting
           if T_plot.size == 0 or P_plot.size == 0 or T_plot.shape != P_plot.shape:
               print(f"  Skipping plot for axis {angle_labels_plot[i]} due to empty or mismatched arrays.")
               axes_test_plot[i, 0].set_title(f'angle_{angle_labels_plot[i]} (No Data)')
               axes_test_plot[i, 1].set_title(f'diff angle_{angle_labels_plot[i]} (No Data)')
               plot_successful = False
               continue

           # Calculate mean and std, handling potential all-NaN slices if necessary
           with np.errstate(invalid='ignore'): # Suppress warnings for mean of empty slice
                mT = np.nanmean(T_plot, axis=0)
                sT = np.nanstd(T_plot, axis=0)
                mP = np.nanmean(P_plot, axis=0)
                sP = np.nanstd(P_plot, axis=0)
                diff_plot = np.abs(T_plot - P_plot)
                mD = np.nanmean(diff_plot, axis=0)
                sD = np.nanstd(diff_plot, axis=0)


           # -------- left column: True vs Pred --------
           ax = axes_test_plot[i, 0]
           ax.plot(time_plot, mT, lw=2, color='teal',   label='True')
           ax.plot(time_plot, mP, lw=2, color='purple', label='Pred')
           ax.fill_between(time_plot, mT - sT, mT + sT, alpha=0.15, color='teal', where=~np.isnan(mT))
           ax.fill_between(time_plot, mP - sP, mP + sP, alpha=0.15, color='purple', where=~np.isnan(mP))
           ax.set_title(f'TEST angle_{angle_labels_plot[i]}')
           if i == 0: ax.legend()
           ax.set_ylabel('Angle (°)')

           # Add grid lines for better readability
           ax.grid(True, linestyle='--', alpha=0.6)


           # -------- right column: |True – Pred| --------
           axd = axes_test_plot[i, 1]
           axd.plot(time_plot, mD, lw=2)
           axd.fill_between(time_plot, mD - sD, mD + sD, alpha=0.25, where=~np.isnan(mD))
           axd.set_title(f'TEST |True – Pred| angle_{angle_labels_plot[i]}')
           axd.set_ylabel('Absolute Error (°)')
           axd.grid(True, linestyle='--', alpha=0.6)
           # Set y-limit starting from 0 for difference plots
           axd.set_ylim(bottom=0)


        axes_test_plot[-1, 0].set_xlabel('Normalized Time (%)')
        axes_test_plot[-1, 1].set_xlabel('Normalized Time (%)')
        fig_test_plot.suptitle('Residual CNN v3 - TEST Set Performance Summary', fontsize=14)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout for suptitle
        pp_test_plot.savefig(fig_test_plot)
        pp_test_plot.close()
        plt.show() # Display inline
        if plot_successful:
             print("  Test plot saved successfully.")
        else:
             print("  Test plot saved, but some axes might be missing data.")

    except Exception as e:
        print(f"  ❌ Error generating or saving TEST plot: {e}")
        # Ensure pdf is closed even on error
        if 'pp_test_plot' in locals() and pp_test_plot. ouvertes:
             pp_test_plot.close()
        plt.close(fig_test_plot) # Close the figure to prevent display issues


In [20]:
## --- Calculate Final TEST Set Metrics for Residual CNN v2 ---
import numpy as np
import pandas as pd
from dataclasses import dataclass, asdict
import math # Make sure math is imported

print("\n" + "="*80)
print("CALCULATING FINAL TEST SET METRICS for Residual CNN v2")
print("="*80)

# ----------------------------- Configuration -------------------------------- #
AXES_METRICS = ('X', 'Y', 'Z')
RANGE_EPS_METRICS = 1e-8    # guard against zero range

# Paths to the aggregated Excel files created by THIS notebook
train_path_for_metrics = join(outputExcelBaseDir, 'TruePredDiff_train.xlsx')
test_path_for_metrics = join(outputExcelBaseDir, 'TruePredDiff.xlsx')

print(f"Reading Training data for ranges from: {train_path_for_metrics}")
print(f"Reading Test data for evaluation from: {test_path_for_metrics}")
print("-" * 60)

# Check if files exist
train_metrics_file_exists = os.path.exists(train_path_for_metrics)
test_metrics_file_exists = os.path.exists(test_path_for_metrics)

if not train_metrics_file_exists:
    print(f"❌ ERROR: Training metrics file not found: {train_path_for_metrics}. Cannot calculate ranges.")
if not test_metrics_file_exists:
    print(f"❌ ERROR: Test metrics file not found: {test_path_for_metrics}. Cannot evaluate test set.")
print("-" * 60)

# ----------------------------- Data Classes (reuse from original) ------------------ #
@dataclass
class AxisMetrics:
    axis: str
    training_range_deg: float
    training_min_deg: float
    training_max_deg: float
    global_rmse_deg: float
    global_nrmse_pct: float
    global_corr: float
    sd_ratio: float
    per_trial_corr_mean: float
    per_trial_corr_std: float
    per_trial_rmse_median_deg: float # Using median/IQR as robust measures
    per_trial_rmse_iqr_deg: float
    per_trial_nrmse_mean_pct: float
    per_trial_nrmse_std_pct: float
    n_trials_total: int
    n_trials_used: int
    n_trials_dropped: int
    # Store per-trial arrays for potential downstream use (like comparing to baseline)
    per_trial_corrs: np.ndarray = None
    per_trial_rmse_deg: np.ndarray = None
    per_trial_nrmse_pct: np.ndarray = None

# ----------------------------- Utility Functions (reuse/adapt) -------------------- #
def load_axis_sheets_metrics(path, axis):
    """Load true & pred sheets for one axis (rows=time, cols=trials)."""
    try:
        # Read Excel, assuming timepoints are index
        t_df = pd.read_excel(path, sheet_name=f"true_{axis}", index_col=0)
        p_df = pd.read_excel(path, sheet_name=f"pred_{axis}", index_col=0)

        # Transpose to get (trials, time) numpy arrays
        t = t_df.transpose().to_numpy()
        p = p_df.transpose().to_numpy()

        # Basic shape check
        if t.shape[0] != p.shape[0] or t.shape[1] != 101 or p.shape[1] != 101:
             print(f"  Warning: Shape mismatch or incorrect timepoints for axis {axis} in {path}.")
             print(f"    True shape: {t.shape}, Pred shape: {p.shape}")
             return np.array([]), np.array([]) # Return empty arrays on error

        return t, p

    except FileNotFoundError:
        print(f"  Error: File not found - {path}")
        return np.array([]), np.array([])
    except ValueError as e:
        print(f"  Error: Sheet not found or other reading error for axis {axis} in {path}: {e}")
        return np.array([]), np.array([])
    except Exception as e:
         print(f"  Error loading sheets for axis {axis} from {path}: {e}")
         return np.array([]), np.array([])

def compute_training_ranges_metrics(train_path, axes=AXES_METRICS):
    """Compute global min/max & range per axis from training set true data."""
    ranges = {}
    print("Computing training data ranges (for normalization)...")
    print("-" * 70)
    all_axes_loaded = True
    for ax in axes:
        # Load only the true data (transposed to trials, time)
        t, _ = load_axis_sheets_metrics(train_path, ax)
        if t.size == 0:
             print(f"  ❌ Failed to load true training data for axis {ax}. Cannot compute range.")
             ranges[ax] = dict(range=np.nan, min=np.nan, max=np.nan) # Mark as invalid
             all_axes_loaded = False
             continue

        # Calculate range, handling potential NaNs
        gmin = np.nanmin(t)
        gmax = np.nanmax(t)
        if np.isnan(gmin) or np.isnan(gmax):
            print(f"  Warning: NaNs encountered in true training data for axis {ax}. Range might be inaccurate.")
            rng = np.nan
        else:
            rng = max(gmax - gmin, RANGE_EPS_METRICS)

        ranges[ax] = dict(range=rng, min=gmin, max=gmax)
        print(f"  {ax}: range={rng:.2f}° (min {gmin:.2f}°, max {gmax:.2f}°)")

    print("-" * 70 + "\n")
    if not all_axes_loaded:
        print("  Warning: Failed to load training data for one or more axes. Subsequent nRMSE calculations may fail.")
    return ranges

def evaluate_axis_metrics(train_info, test_path, axis):
    """
    Evaluate model predictions for one axis on the *test* set using training range.
    Returns AxisMetrics.
    """
    training_range = train_info.get('range', np.nan) # Get range, default to NaN if missing
    t_min = train_info.get('min', np.nan)
    t_max = train_info.get('max', np.nan)

    if np.isnan(training_range):
         print(f"  Skipping evaluation for axis {axis} due to missing training range.")
         # Return a default/error structure
         return AxisMetrics(axis=axis, n_trials_total=0, n_trials_used=0, n_trials_dropped=0,
                            training_range_deg=np.nan, training_min_deg=np.nan, training_max_deg=np.nan,
                            global_rmse_deg=np.nan, global_nrmse_pct=np.nan, global_corr=np.nan, sd_ratio=np.nan,
                            per_trial_corr_mean=np.nan, per_trial_corr_std=np.nan,
                            per_trial_rmse_median_deg=np.nan, per_trial_rmse_iqr_deg=np.nan,
                            per_trial_nrmse_mean_pct=np.nan, per_trial_nrmse_std_pct=np.nan)


    T, P = load_axis_sheets_metrics(test_path, axis) # Shape (trials, time)

    if T.size == 0 or P.size == 0:
         print(f"  Skipping evaluation for axis {axis} due to missing test data.")
         return AxisMetrics(axis=axis, n_trials_total=0, n_trials_used=0, n_trials_dropped=0,
                           training_range_deg=training_range, training_min_deg=t_min, training_max_deg=t_max,
                           # Fill remaining fields with NaN or appropriate defaults
                            global_rmse_deg=np.nan, global_nrmse_pct=np.nan, global_corr=np.nan, sd_ratio=np.nan,
                            per_trial_corr_mean=np.nan, per_trial_corr_std=np.nan,
                            per_trial_rmse_median_deg=np.nan, per_trial_rmse_iqr_deg=np.nan,
                            per_trial_nrmse_mean_pct=np.nan, per_trial_nrmse_std_pct=np.nan)


    # Identify & drop any trial with a NaN in true or pred
    invalid = np.isnan(T).any(axis=1) | np.isnan(P).any(axis=1)
    valid_mask = ~invalid
    T_valid = T[valid_mask, :] # Shape (n_valid, time)
    P_valid = P[valid_mask, :]

    n_total = T.shape[0]
    n_dropped = int(invalid.sum())
    n_used = int(valid_mask.sum())

    if n_used == 0:
        print(f"  Axis {axis}: No valid trials after NaN filtering.")
        return AxisMetrics(axis=axis, n_trials_total=n_total, n_trials_used=0, n_trials_dropped=n_dropped,
                           training_range_deg=training_range, training_min_deg=t_min, training_max_deg=t_max,
                           # Fill remaining fields with NaN
                            global_rmse_deg=np.nan, global_nrmse_pct=np.nan, global_corr=np.nan, sd_ratio=np.nan,
                            per_trial_corr_mean=np.nan, per_trial_corr_std=np.nan,
                            per_trial_rmse_median_deg=np.nan, per_trial_rmse_iqr_deg=np.nan,
                            per_trial_nrmse_mean_pct=np.nan, per_trial_nrmse_std_pct=np.nan)

    # Per-trial metrics
    per_trial_corrs = []
    per_trial_rmse = []
    for j in range(n_used):
        t_true = T_valid[j, :]
        t_pred = P_valid[j, :]

        # Skip if all NaNs (shouldn't happen with filter, but safe)
        if np.all(np.isnan(t_true)) or np.all(np.isnan(t_pred)):
             per_trial_corrs.append(np.nan)
             per_trial_rmse.append(np.nan)
             continue

        # Correlation (guard zero variance)
        if np.nanstd(t_true) > RANGE_EPS_METRICS and np.nanstd(t_pred) > RANGE_EPS_METRICS:
            c = np.corrcoef(t_true, t_pred)[0, 1]
        else:
            c = np.nan
        per_trial_corrs.append(c)
        per_trial_rmse.append(np.sqrt(np.nanmean((t_pred - t_true)**2))) # Use nanmean

    per_trial_corrs = np.array(per_trial_corrs, dtype=float)
    per_trial_rmse = np.array(per_trial_rmse, dtype=float)

    # Filter NaNs from results
    valid_corr_mask = ~np.isnan(per_trial_corrs)
    valid_rmse_mask = ~np.isnan(per_trial_rmse)
    per_trial_corrs_clean = per_trial_corrs[valid_corr_mask]
    per_trial_rmse_clean = per_trial_rmse[valid_rmse_mask]

    per_trial_nrmse_pct_clean = 100.0 * per_trial_rmse_clean / training_range

    # Global metrics using only valid trials
    flat_true = T_valid.reshape(-1)
    flat_pred = P_valid.reshape(-1)
    mask = ~np.isnan(flat_true) & ~np.isnan(flat_pred)
    flat_true = flat_true[mask]
    flat_pred = flat_pred[mask]

    global_rmse = np.sqrt(np.mean((flat_pred - flat_true)**2)) if mask.any() else np.nan
    if flat_true.size > 1 and np.std(flat_true) > RANGE_EPS_METRICS and np.std(flat_pred) > RANGE_EPS_METRICS:
        global_corr = np.corrcoef(flat_true, flat_pred)[0, 1]
    else:
        global_corr = np.nan
    global_nrmse_pct = 100.0 * global_rmse / training_range if not np.isnan(global_rmse) else np.nan

    # SD ratio (across-trial dispersion at each timepoint)
    # T_valid/P_valid are (n_valid, time) -> axis=0 for std across trials
    true_sd_time = np.nanstd(T_valid, axis=0)
    pred_sd_time = np.nanstd(P_valid, axis=0)
    nz = true_sd_time > RANGE_EPS_METRICS
    if nz.any():
        # Ensure pred_sd_time[nz] is not all zero before division
        if np.any(pred_sd_time[nz]):
            sd_ratio = np.nanmean(pred_sd_time[nz] / true_sd_time[nz])
        else:
            sd_ratio = 0.0 # If pred is constant but true varies
    else:
        sd_ratio = np.nan # If true signal is constant across all trials


    # Robust summaries for per-trial RMSE
    rmse_median = float(np.nanmedian(per_trial_rmse_clean)) if per_trial_rmse_clean.size else np.nan
    rmse_iqr = float(np.nanpercentile(per_trial_rmse_clean, 75) - np.nanpercentile(per_trial_rmse_clean, 25)) if per_trial_rmse_clean.size else np.nan

    return AxisMetrics(
        axis=axis,
        training_range_deg=training_range,
        training_min_deg=t_min,
        training_max_deg=t_max,
        global_rmse_deg=global_rmse,
        global_nrmse_pct=global_nrmse_pct,
        global_corr=global_corr,
        sd_ratio=sd_ratio,
        per_trial_corr_mean=float(np.nanmean(per_trial_corrs_clean)) if per_trial_corrs_clean.size else np.nan,
        per_trial_corr_std=float(np.nanstd(per_trial_corrs_clean)) if per_trial_corrs_clean.size else np.nan,
        per_trial_rmse_median_deg=rmse_median,
        per_trial_rmse_iqr_deg=rmse_iqr,
        per_trial_nrmse_mean_pct=float(np.nanmean(per_trial_nrmse_pct_clean)) if per_trial_nrmse_pct_clean.size else np.nan,
        per_trial_nrmse_std_pct=float(np.nanstd(per_trial_nrmse_pct_clean)) if per_trial_nrmse_pct_clean.size else np.nan,
        n_trials_total=n_total,
        n_trials_used=n_used,
        n_trials_dropped=n_dropped,
        per_trial_corrs=per_trial_corrs, # Store original array including potential NaNs
        per_trial_rmse_deg=per_trial_rmse,
        per_trial_nrmse_pct=100.0 * per_trial_rmse / training_range # Calculate nRMSE including NaNs if rmse was NaN
    )

def print_axis_table_metrics(metrics_list, dataset_name="TEST SET"):
    """Pretty print per-axis summary."""
    print("\n" + "=" * 95)
    print(f"{dataset_name.upper()} EVALUATION (normalized by training ranges)")
    print("=" * 95)
    header = ("Axis  "
              "Per-trial Corr (μ±σ)  "
              "GlobalCorr  "
              "GlobRMSE°  Glob nRMSE%  "
              "SD_ratio  "
              "Per-trial nRMSE μ±σ (%)  "
              "TrialsUsed/Total")
    print(header)
    print("-" * 95)
    valid_metrics = [m for m in metrics_list if not np.isnan(m.global_corr)] # Filter out axes with errors
    if not valid_metrics:
         print("  No valid metrics to display.")
         print("-" * 95)
         return

    for m in valid_metrics:
        corr_mean_str = f"{m.per_trial_corr_mean:>6.3f}" if not np.isnan(m.per_trial_corr_mean) else "  nan "
        corr_std_str = f"{m.per_trial_corr_std:>5.3f}" if not np.isnan(m.per_trial_corr_std) else " nan "
        glob_corr_str = f"{m.global_corr:>9.3f}" if not np.isnan(m.global_corr) else "   nan   "
        glob_rmse_str = f"{m.global_rmse_deg:>7.3f}" if not np.isnan(m.global_rmse_deg) else "  nan  "
        glob_nrmse_str = f"{m.global_nrmse_pct:>7.2f}" if not np.isnan(m.global_nrmse_pct) else "  nan  "
        sd_ratio_str = f"{m.sd_ratio:>7.3f}" if not np.isnan(m.sd_ratio) else "  nan  "
        pt_nrmse_mean_str = f"{m.per_trial_nrmse_mean_pct:>6.2f}" if not np.isnan(m.per_trial_nrmse_mean_pct) else " nan "
        pt_nrmse_std_str = f"{m.per_trial_nrmse_std_pct:>5.2f}" if not np.isnan(m.per_trial_nrmse_std_pct) else " nan "

        print(f"{m.axis:<4} "
              f"{corr_mean_str}±{corr_std_str}  "
              f"{glob_corr_str}  "
              f"{glob_rmse_str}   {glob_nrmse_str}   "
              f"{sd_ratio_str}  "
              f"{pt_nrmse_mean_str}±{pt_nrmse_std_str}   "
              f"{m.n_trials_used:>3}/{m.n_trials_total}")
    print("-" * 95)
    # Macro averages (ignore NaNs)
    gm = [m.global_rmse_deg for m in valid_metrics]
    gn = [m.global_nrmse_pct for m in valid_metrics]
    gc = [m.global_corr for m in valid_metrics]
    sr = [m.sd_ratio for m in valid_metrics]
    macro_rmse = np.nanmean(gm)
    macro_nrmse = np.nanmean(gn)
    macro_corr = np.nanmean(gc)
    macro_sd_ratio = np.nanmean(sr)
    print(f"AVG   global_corr={macro_corr:.3f}  global_rmse={macro_rmse:.3f}°  "
          f"global_nRMSE={macro_nrmse:.2f}%  SD_ratio={macro_sd_ratio:.3f}")
    print("=" * 95)
    print("Notes:")
    print("  • SD_ratio <1 indicates under-dispersion (shrinkage); >1 over-dispersion; ≈1 well-calibrated variance.")
    print("  • global_nRMSE uses a *single* training min-max per axis; per-trial nRMSE is mean of each trial's RMSE / training range.")
    print("  • Use stored per_trial_rmse_deg arrays (e.g., test_metrics[0].per_trial_rmse_deg) for ΔRMSE vs naive baseline.")
    print()

def main_metrics(train_path, test_path, axes):
    # 1. Compute training ranges
    if not train_metrics_file_exists: return [], None, None # Cannot proceed without ranges
    training_ranges = compute_training_ranges_metrics(train_path, axes)

    # 2. Evaluate each axis on test
    test_metrics = []
    if not test_metrics_file_exists:
         print("Skipping test set evaluation as file is missing.")
    else:
        for ax in axes:
            m = evaluate_axis_metrics(training_ranges.get(ax, {}), test_path, ax)
            test_metrics.append(m)

    # 3. Print test table
    print_axis_table_metrics(test_metrics, dataset_name="TEST SET")

    # 4. Optional: Save summary (can be done outside if needed)
    # ...

    return test_metrics, training_ranges

# --- Execute Test Set Metric Calculation ---
if train_metrics_file_exists and test_metrics_file_exists:
     test_metrics_results, calculated_training_ranges = main_metrics(train_path_for_metrics, test_path_for_metrics, AXES_METRICS)
     # Store per-trial baseline RMSEs if baseline ran successfully
     if naive_results and "all_per_trial_rmse_baseline" in naive_results:
          baseline_per_trial_rmse = naive_results["all_per_trial_rmse_baseline"]
          print("\nBaseline per-trial RMSEs stored for comparison.")
     else:
          baseline_per_trial_rmse = None
          print("\nWarning: Naive baseline results not available for comparison.")
else:
     print("\nSkipping final TEST SET metric calculation due to missing Excel files.")
     test_metrics_results = []
     calculated_training_ranges = None
     baseline_per_trial_rmse = None


CALCULATING FINAL TEST SET METRICS for Residual CNN v2
Reading Training data for ranges from: R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Training_results\anglePyramidCNN_RESIDUAL_v3\IWALQQ_1st_correction\angle\TruePredDiff_train.xlsx
Reading Test data for evaluation from: R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Training_results\anglePyramidCNN_RESIDUAL_v3\IWALQQ_1st_correction\angle\TruePredDiff.xlsx
------------------------------------------------------------
------------------------------------------------------------
Computing training data ranges (for normalization)...
----------------------------------------------------------------------
  X: range=75.98° (min -67.41°, max 8.56°)
  Y: range=27.80° (min -13.18°, max 14.61°)
  Z: range=55.54° (min -41.26°, max 14.28°)
----------------------------------------------------------------------


TEST SET EVALUATION (normalized by training ranges)
Axis  Per-trial Corr (μ±σ)  GlobalC

# Analysis on Training Data
Now repeat the inference and analysis steps using the model's predictions on the training data.

In [15]:
# --- ENSEMBLE INFERENCE CODE - Adapted for Residual CNN v2 on TRAINING Data ---
from os.path import join
import os
import numpy as np
import pandas as pd
import torch
from pickle import load

print("\n" + "="*80)
print("PERFORMING INFERENCE ON TRAINING SET using Residual CNN v2 Ensemble")
print("="*80)

# Ensure necessary variables are defined (should be from Cell 5 and Cell 7)
# MODELS_DIR, DATASET_DIR, outputExcelBaseDir, device
# BEST_PARAMS_USED (loaded in Cell 7)

total_train_files_processed = 0
for numFold in range(5):
    print(f"\n--- Processing Fold {numFold} (Training Data) ---")
    # 1. Load ensemble checkpoint
    ensemble_path = join(MODELS_DIR, f'fold_{numFold}_ensemble_models.pt')
    if not os.path.exists(ensemble_path):
        print(f"  Ensemble file not found: {ensemble_path}. Skipping fold.")
        continue

    try:
        ckpt = torch.load(ensemble_path, map_location=torch.device(device))
        state_dicts = ckpt.get('models')
        ensemble_config = ckpt.get('config', BEST_PARAMS_USED)
        model_seeds = ckpt.get('seeds', ['unknown'] * len(state_dicts) if state_dicts else [])
        if state_dicts is None:
             print(f"  ❌ Key 'models' not found in checkpoint: {ensemble_path}. Skipping fold.")
             continue
        print(f"  Fold {numFold}: loaded ensemble with {len(state_dicts)} models (Seeds: {model_seeds}).")
    except Exception as e:
        print(f"  ❌ Error loading checkpoint {ensemble_path}: {e}. Skipping fold.")
        continue

    # 2. Extract CNN kwargs (reuse logic from test inference)
    try:
        cnn_kwargs = {
            'kernels':      ensemble_config['kernels'],
            'channels':     ensemble_config['channels'],
            'n_layers':     ensemble_config['n_layers'],
            'dropout_conv': ensemble_config['dropout_conv'],
            'dropout_fc':   ensemble_config['dropout_fc'],
            'num_heads':    ensemble_config.get('num_heads', 4),
        }
    except KeyError as e:
        print(f"  ❌ Missing key '{e}' in loaded ensemble config for fold {numFold}. Using default BEST_PARAMS.")
        try: # Fallback
            cnn_kwargs = {
                'kernels':      BEST_PARAMS['kernels'], 'channels':     BEST_PARAMS['channels'],
                'n_layers':     BEST_PARAMS['n_layers'], 'dropout_conv': BEST_PARAMS['dropout_conv'],
                'dropout_fc':   BEST_PARAMS['dropout_fc'], 'num_heads':    BEST_PARAMS.get('num_heads', 4),
            }
        except KeyError as e_fallback:
             print(f"  ❌ Missing key '{e_fallback}' in fallback BEST_PARAMS. Cannot build model. Skipping fold.")
             continue

    # 3. Rebuild ensemble models (reuse logic)
    ensemble_models = []
    models_loaded_successfully = True
    for idx, sd in enumerate(state_dicts):
        try:
            m = PyramidAttnCNN(**cnn_kwargs)
            m.load_state_dict(sd)
            m.to(device)
            m.eval()
            ensemble_models.append(m)
        except Exception as e:
            print(f"  ❌ Error loading state dict for model {idx} in fold {numFold}: {e}")
            models_loaded_successfully = False
            break

    if not models_loaded_successfully or not ensemble_models:
        print(f"  Skipping inference for fold {numFold} due to model loading errors.")
        continue

    # 4. Load TRAINING data features + output scaler
    load_train_path = join(DATASET_DIR, f"{numFold}_fold_final_train.npz") # <-- Use _train NPZ
    scaler_path = join(DATASET_DIR, f"{numFold}_fold_scaler4Y_{DATA_TYPE}.pkl")

    if not os.path.exists(load_train_path) or not os.path.exists(scaler_path):
        print(f"  ❌ Train NPZ or Scaler PKL file missing for fold {numFold}. Skipping.")
        continue

    try:
        load_train_data = np.load(load_train_path)
        x_train_key = "final_X_train" # <-- Use _train key
        y_train_key = f"final_Y_{DATA_TYPE}_train" # <-- Use _train key
        if x_train_key not in load_train_data or y_train_key not in load_train_data:
             print(f"  ❌ Keys '{x_train_key}' or '{y_train_key}' not found in {load_train_path}. Skipping fold.")
             continue
        X_train_all = load_train_data[x_train_key]
        Y_true_scaled_all = load_train_data[y_train_key]

        with open(scaler_path, "rb") as f:
             load_scaler4Y = load(f) # Add encoding if needed

        n_samples = X_train_all.shape[0]
        print(f"  Processing fold {numFold} with {n_samples} training samples")

    except Exception as e:
        print(f"  ❌ Error loading training data or scaler for fold {numFold}: {e}. Skipping.")
        continue

    # 5. Inference loop for training data
    fold_files_processed_train = 0
    for dataIndex in range(n_samples):
        X_sample = X_train_all[dataIndex].reshape(1, NUM_FEATURES, SEQ_LEN)
        Y_true_scaled_sample = Y_true_scaled_all[dataIndex]

        x_tensor = torch.from_numpy(X_sample).float().to(device)

        with torch.no_grad():
            preds_scaled = []
            for model in ensemble_models:
                pred_scaled = model(x_tensor)
                preds_scaled.append(pred_scaled.cpu().numpy())

            Y_pred_scaled_avg = np.mean(np.stack(preds_scaled, axis=0), axis=0)

        Y_pred_scaled_reshaped = Y_pred_scaled_avg.reshape(3, -1).T
        Y_true_scaled_reshaped = Y_true_scaled_sample.reshape(3, -1).T

        # Un-scale
        try:
            if load_scaler4Y.min_.shape != (3,) or load_scaler4Y.scale_.shape != (3,):
                 print(f"    ❌ Scaler shape mismatch for fold {numFold} during unscaling (train). Skipping sample {dataIndex}.")
                 continue
            Y_true_unscaled = (Y_true_scaled_reshaped - load_scaler4Y.min_) / load_scaler4Y.scale_
            Y_pred_unscaled = (Y_pred_scaled_reshaped - load_scaler4Y.min_) / load_scaler4Y.scale_
        except Exception as e:
            print(f"    ❌ Error unscaling training data for sample {dataIndex}, fold {numFold}: {e}. Skipping sample.")
            continue

        # Save to Excel in "_train" directory
        df_true = pd.DataFrame(Y_true_unscaled, columns=['X_True','Y_True','Z_True'])
        df_pred = pd.DataFrame(Y_pred_unscaled, columns=['X_Pred','Y_Pred','Z_Pred'])
        df_save = pd.concat([df_true, df_pred], axis=1)

        out_dir_fold_train = join(outputExcelBaseDir, f"{numFold}_fold_train") # <-- Save to _train folder
        ensure_dir(out_dir_fold_train)

        excel_filename = f"{dataIndex}.xlsx"
        excel_filepath = join(out_dir_fold_train, excel_filename)
        try:
            df_save.to_excel(excel_filepath, index=True, index_label="Timepoint")
            fold_files_processed_train += 1
        except Exception as e:
            print(f"    ❌ Error saving Excel file {excel_filepath}: {e}")

    print(f"  Fold {numFold} (Training Data) completed successfully! Processed {fold_files_processed_train}/{n_samples} samples.")
    total_train_files_processed += fold_files_processed_train

print("\n" + "="*80)
print(f"All Residual CNN v2 TRAINING folds processed. Total files saved: {total_train_files_processed}")
print("="*80)


PERFORMING INFERENCE ON TRAINING SET using Residual CNN v2 Ensemble

--- Processing Fold 0 (Training Data) ---


C:\Users\asmith8\AppData\Local\Temp\ipykernel_3168\4137637000.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ensemble_path, map_location=torch.device

  Fold 0: loaded ensemble with 3 models (Seeds: [42, 123, 456]).


C:\Users\asmith8\AppData\Local\anaconda3\envs\imu_dl\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  Processing fold 0 with 722 training samples
  Fold 0 (Training Data) completed successfully! Processed 722/722 samples.

--- Processing Fold 1 (Training Data) ---


...

  Processing fold 3 with 670 training samples
  Fold 3 (Training Data) completed successfully! Processed 670/670 samples.

--- Processing Fold 4 (Training Data) ---


C:\Users\asmith8\AppData\Local\Temp\ipykernel_3168\4137637000.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ensemble_path, map_location=torch.device

  Fold 4: loaded ensemble with 3 models (Seeds: [42, 123, 456]).


C:\Users\asmith8\AppData\Local\anaconda3\envs\imu_dl\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  Processing fold 4 with 723 training samples
  Fold 4 (Training Data) completed successfully! Processed 723/723 samples.

All Residual CNN v2 TRAINING folds processed. Total files saved: 3504


In [16]:
# --- Aggregate TRAINING Set Results from Excel Files ---
print("\nAggregating TRAINING set results from Excel files...")
# Initialize empty numpy arrays for training data
true_train_X = np.array([])
pred_train_X = np.array([])
true_train_Y = np.array([])
pred_train_Y = np.array([])
true_train_Z = np.array([])
pred_train_Z = np.array([])

train_folders_agg = [f"{fold}_fold_train" for fold in range(5)] # <-- Use _train folder names
total_train_files_loaded_agg = 0
problematic_files_train = []

for fold in train_folders_agg:
    fold_path = join(outputExcelBaseDir, fold) # outputExcelBaseDir is outputExcelBaseDir
    if not os.path.exists(fold_path):
        print(f"  Skipping non-existent train fold for aggregation: {fold}")
        continue
    print(f"  Aggregating from train fold: {fold}")
    list_results = natsorted([f for f in os.listdir(fold_path) if f.endswith(".xlsx")])
    fold_files_loaded = 0
    for result_file in list_results:
        file_path = join(fold_path, result_file)
        try:
            singleResult = pd.read_excel(file_path, index_col=0)

            if singleResult.shape[0] != 101:
                print(f"    Warning: File {result_file} does not have 101 rows (shape: {singleResult.shape}). Skipping.")
                problematic_files_train.append(file_path)
                continue

            loaded_axes_in_file = 0
            temp_true_train = {}
            temp_pred_train = {}
            valid_file = True
            for axe in ["X", "Y", "Z"]:
                true_col = f"{axe}_True"
                pred_col = f"{axe}_Pred"
                if true_col in singleResult.columns and pred_col in singleResult.columns:
                    true_vals = singleResult[true_col].to_numpy()
                    pred_vals = singleResult[pred_col].to_numpy()

                    if np.isnan(true_vals).any() or np.isnan(pred_vals).any():
                         print(f"    Warning: NaNs found in axis {axe} of {result_file}. Skipping file.")
                         problematic_files_train.append(file_path + f" (NaNs in axis {axe})")
                         valid_file = False
                         break

                    temp_true_train[axe] = np.expand_dims(true_vals, axis=0)
                    temp_pred_train[axe] = np.expand_dims(pred_vals, axis=0)
                    loaded_axes_in_file += 1
                else:
                    print(f"    Warning: Missing columns for axis {axe} in {result_file}. Skipping file.")
                    problematic_files_train.append(file_path + f" (Missing cols for axis {axe})")
                    valid_file = False
                    break

            if valid_file and loaded_axes_in_file == 3:
                for axe in ["X", "Y", "Z"]:
                    current_true_train = globals().get(f"true_train_{axe}")
                    current_pred_train = globals().get(f"pred_train_{axe}")
                    if current_true_train is not None and current_true_train.size == 0:
                        globals()[f"true_train_{axe}"] = temp_true_train[axe]
                        globals()[f"pred_train_{axe}"] = temp_pred_train[axe]
                    elif current_true_train is not None:
                        globals()[f"true_train_{axe}"] = np.concatenate((current_true_train, temp_true_train[axe]), 0)
                        globals()[f"pred_train_{axe}"] = np.concatenate((current_pred_train, temp_pred_train[axe]), 0)
                fold_files_loaded += 1
            elif valid_file and loaded_axes_in_file != 3:
                 print(f"    Warning: File {result_file} loaded successfully but didn't contain all 3 axes (train). Skipping aggregation.")
                 problematic_files_train.append(file_path + f" (Incomplete axes)")


        except Exception as e:
            print(f"    ❌ Error aggregating file {result_file}: {e}")
            problematic_files_train.append(file_path + f" (Read Error: {e})")

    print(f"    Aggregated {fold_files_loaded} valid files from {fold}")
    total_train_files_loaded_agg += fold_files_loaded

print(f"\nTotal TRAINING files aggregated: {total_train_files_loaded_agg}")
print("Aggregated training array shapes:")
print(f"  True Train X: {true_train_X.shape}, Pred Train X: {pred_train_X.shape}")
print(f"  True Train Y: {true_train_Y.shape}, Pred Train Y: {pred_train_Y.shape}")
print(f"  True Train Z: {true_train_Z.shape}, Pred Train Z: {pred_train_Z.shape}")

if problematic_files_train:
     print("\nProblematic files encountered during TRAINING aggregation:")
     for f in problematic_files_train[:10]: # Print first 10 issues
          print(f"  - {f}")
     if len(problematic_files_train) > 10:
          print(f"  ... and {len(problematic_files_train)-10} more.")

print("\n✅ Training set aggregation complete.")


Aggregating TRAINING set results from Excel files...
  Aggregating from train fold: 0_fold_train
    Aggregated 722 valid files from 0_fold_train
  Aggregating from train fold: 1_fold_train
    Aggregated 667 valid files from 1_fold_train
  Aggregating from train fold: 2_fold_train
    Aggregated 722 valid files from 2_fold_train
  Aggregating from train fold: 3_fold_train
    Aggregated 670 valid files from 3_fold_train
  Aggregating from train fold: 4_fold_train
    Aggregated 723 valid files from 4_fold_train

Total TRAINING files aggregated: 3504
Aggregated training array shapes:
  True Train X: (3504, 101), Pred Train X: (3504, 101)
  True Train Y: (3504, 101), Pred Train Y: (3504, 101)
  True Train Z: (3504, 101), Pred Train Z: (3504, 101)

✅ Training set aggregation complete.


In [17]:
# --- Save Aggregated TRAINING Data to Multi-Sheet Excel ---
# Ensure xlsxwriter is imported/installed (done in Cell 13)

train_excel_path_save = join(outputExcelBaseDir,'TruePredDiff_train.xlsx') # outputExcelBaseDir is outputExcelBaseDir
print(f"\nSaving aggregated TRAINING results to: {train_excel_path_save}")

# Check if training data exists before writing
if 'true_train_X' not in locals() or true_train_X.size == 0:
    print("  No aggregated TRAINING data available to save. Skipping Excel write.")
else:
    try:
        writer_save_train = pd.ExcelWriter(train_excel_path_save, engine='xlsxwriter')
        write_success_train = True
        for axe in ["X", "Y", "Z"]:
            for sess in ["true", "pred"]:
                # Use the correct variable names with "_train"
                dataName_train = f"{sess}_train_{axe}"
                if dataName_train in globals() and globals()[dataName_train].size > 0:
                     df_to_save_train = makeDataframe(globals()[dataName_train])
                     if not df_to_save_train.empty:
                          # Save with simplified sheet names for metric function compatibility
                          sheet_name_simple_train = f"{sess}_{axe}"
                          df_to_save_train.to_excel(writer_save_train, sheet_name=sheet_name_simple_train, index=True, index_label="Timepoint") # Save index
                     else:
                          print(f"  Skipping empty train sheet: {dataName_train}")
                else:
                    print(f"  Warning: Training data array '{dataName_train}' not found or empty. Skipping sheet.")
                    write_success_train = False

            # Calculate and save differences using the correct variable names
            true_train_data_save = globals().get(f"true_train_{axe}")
            pred_train_data_save = globals().get(f"pred_train_{axe}")
            if true_train_data_save is not None and pred_train_data_save is not None and \
               true_train_data_save.size > 0 and pred_train_data_save.size > 0 and \
               true_train_data_save.shape == pred_train_data_save.shape:
                df_diff_train_save = makeDataframe(abs(true_train_data_save - pred_train_data_save))
                if not df_diff_train_save.empty:
                    df_diff_train_save.to_excel(writer_save_train, sheet_name=str(f"diff_{axe}"), index=True, index_label="Timepoint")
                else:
                    print(f"  Skipping empty train diff sheet: diff_{axe}")
            else:
                 print(f"  Warning: Cannot calculate or save diff for training axis {axe} due to missing/mismatched data.")
                 write_success_train = False

        writer_save_train.close()
        if write_success_train:
             print("  Training data Excel file saved successfully.")
        else:
             print("  Training data Excel file saved, but some sheets might be missing.")

    except Exception as e:
        print(f"  ❌ Error writing TRAINING Excel file {train_excel_path_save}: {e}")


Saving aggregated TRAINING results to: R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Training_results\anglePyramidCNN_RESIDUAL_v3\IWALQQ_1st_correction\angle\TruePredDiff_train.xlsx
  Training data Excel file saved successfully.


In [ ]:
# --- Generate TRAINING Set Summary Plot ---
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import matplotlib.pyplot as plt

# Check if aggregated training data exists
if 'true_train_X' not in locals() or true_train_X.size == 0:
    print("\nSkipping TRAINING set summary plot generation (no aggregated data).")
else:
    pdf_path_train_plot = join(outputExcelBaseDir, f"WHOLE_Total_result_RESIDUAL_v3_TRAIN_{DATA_TYPE}.pdf")
    print(f"\nGenerating TRAINING set summary plot: {pdf_path_train_plot}")
    try:
        pp_train_plot = PdfPages(pdf_path_train_plot)
        fig_train_plot, axes_train_plot = plt.subplots(3, 2, figsize=(12, 8), sharex=True)
        time_plot_train = np.linspace(0, 100, 101) # X-axis represents % of cycle or normalized time
        angle_labels_train_plot = ['X', 'Y', 'Z']

        plot_successful_train = True
        for i, (T_tr_plot, P_tr_plot) in enumerate(zip([true_train_X, true_train_Y, true_train_Z], [pred_train_X, pred_train_Y, pred_train_Z])):
            # Double-check arrays before plotting
            if T_tr_plot.size == 0 or P_tr_plot.size == 0 or T_tr_plot.shape != P_tr_plot.shape:
               print(f"  Skipping plot for training axis {angle_labels_train_plot[i]} due to empty or mismatched arrays.")
               axes_train_plot[i, 0].set_title(f'angle_{angle_labels_train_plot[i]} (No Data)')
               axes_train_plot[i, 1].set_title(f'diff angle_{angle_labels_train_plot[i]} (No Data)')
               plot_successful_train = False
               continue

            # Calculate mean and std
            with np.errstate(invalid='ignore'):
                mT_tr = np.nanmean(T_tr_plot, axis=0)
                sT_tr = np.nanstd(T_tr_plot, axis=0)
                mP_tr = np.nanmean(P_tr_plot, axis=0)
                sP_tr = np.nanstd(P_tr_plot, axis=0)
                diff_tr_plot = np.abs(T_tr_plot - P_tr_plot)
                mD_tr = np.nanmean(diff_tr_plot, axis=0)
                sD_tr = np.nanstd(diff_tr_plot, axis=0)

            # -------- left column: True vs Pred --------
            ax = axes_train_plot[i, 0]
            ax.plot(time_plot_train, mT_tr, lw=2, color='teal',   label='True')
            ax.plot(time_plot_train, mP_tr, lw=2, color='purple', label='Pred')
            ax.fill_between(time_plot_train, mT_tr - sT_tr, mT_tr + sT_tr, alpha=0.15, color='teal', where=~np.isnan(mT_tr))
            ax.fill_between(time_plot_train, mP_tr - sP_tr, mP_tr + sP_tr, alpha=0.15, color='purple', where=~np.isnan(mP_tr))
            ax.set_title(f'TRAIN angle_{angle_labels_train_plot[i]}')
            if i == 0: ax.legend()
            ax.set_ylabel('Angle (°)')

            ax.grid(True, linestyle='--', alpha=0.6)

            # -------- right column: |True – Pred| --------
            axd = axes_train_plot[i, 1]
            axd.plot(time_plot_train, mD_tr, lw=2)
            axd.fill_between(time_plot_train, mD_tr - sD_tr, mD_tr + sD_tr, alpha=0.25, where=~np.isnan(mD_tr))
            axd.set_title(f'TRAIN |True – Pred| angle_{angle_labels_train_plot[i]}')
            axd.set_ylabel('Absolute Error (°)')
            axd.grid(True, linestyle='--', alpha=0.6)
            axd.set_ylim(bottom=0)

        axes_train_plot[-1, 0].set_xlabel('Normalized Time (%)')
        axes_train_plot[-1, 1].set_xlabel('Normalized Time (%)')
        fig_train_plot.suptitle('Residual CNN v3 - TRAINING Set Performance Summary', fontsize=14)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        pp_train_plot.savefig(fig_train_plot)
        pp_train_plot.close()
        plt.show() # Display inline
        if plot_successful_train:
             print("  Training plot saved successfully.")
        else:
             print("  Training plot saved, but some axes might be missing data.")

    except Exception as e:
        print(f"  ❌ Error generating or saving TRAINING plot: {e}")
        if 'pp_train_plot' in locals() and pp_train_plot.ouvrtes:
             pp_train_plot.close()
        plt.close(fig_train_plot)


In [19]:
# --- Calculate Final TRAINING Set Metrics for Residual CNN v2 ---
import numpy as np
import pandas as pd
from dataclasses import dataclass, asdict
import math # Make sure math is imported

print("\n" + "="*80)
print("CALCULATING FINAL TRAINING SET METRICS for Residual CNN v2")
print("="*80)

# Re-use functions from Cell 15:
# compute_training_ranges_metrics, evaluate_axis_metrics, print_axis_table_metrics

# Paths to the aggregated Excel files
train_path_for_train_metrics = join(outputExcelBaseDir, 'TruePredDiff_train.xlsx') # Use the train excel file for BOTH ranges and evaluation

print(f"Reading Training data for ranges AND evaluation from: {train_path_for_train_metrics}")
print("-" * 60)

train_metrics_file_exists_check = os.path.exists(train_path_for_train_metrics)
if not train_metrics_file_exists_check:
    print(f"❌ ERROR: Training metrics file not found: {train_path_for_train_metrics}. Cannot evaluate training set.")
print("-" * 60)

def main_metrics_train(train_path, axes):
    # 1. Compute training ranges (used for normalization AND evaluation source)
    if not train_metrics_file_exists_check: return [], None
    training_ranges = compute_training_ranges_metrics(train_path, axes)

    # 2. Evaluate each axis on TRAINING data using TRAINING ranges
    train_metrics = []
    for ax in axes:
        # Pass the same training_path for both range info and evaluation data
        m = evaluate_axis_metrics(training_ranges.get(ax, {}), train_path, ax)
        train_metrics.append(m)

    # 3. Print training table
    print_axis_table_metrics(train_metrics, dataset_name="TRAINING SET")

    return train_metrics, training_ranges

# --- Execute Training Set Metric Calculation ---
if train_metrics_file_exists_check:
     train_metrics_results, calculated_training_ranges_for_train = main_metrics_train(train_path_for_train_metrics, AXES_METRICS)
else:
     print("\nSkipping final TRAINING SET metric calculation due to missing Excel file.")
     train_metrics_results = []
     calculated_training_ranges_for_train = None


CALCULATING FINAL TRAINING SET METRICS for Residual CNN v2
Reading Training data for ranges AND evaluation from: R:\KumarLab3\PROJECTS\wesens\Data\Analysis\smith_dl\IMU Deep Learning\Training_results\anglePyramidCNN_RESIDUAL_v3\IWALQQ_1st_correction\angle\TruePredDiff_train.xlsx
------------------------------------------------------------
------------------------------------------------------------
Computing training data ranges (for normalization)...
----------------------------------------------------------------------
  X: range=75.98° (min -67.41°, max 8.56°)
  Y: range=27.80° (min -13.18°, max 14.61°)
  Z: range=55.54° (min -41.26°, max 14.28°)
----------------------------------------------------------------------


TRAINING SET EVALUATION (normalized by training ranges)
Axis  Per-trial Corr (μ±σ)  GlobalCorr  GlobRMSE°  Glob nRMSE%  SD_ratio  Per-trial nRMSE μ±σ (%)  TrialsUsed/Total
-----------------------------------------------------------------------------------------------


In [ ]:
import json
import os
from os.path import join
from pickle import load as pickle_load

import numpy as np
from sklearn.linear_model import Ridge


def compute_metrics(y_true, y_pred, train_range, eps=1e-8):
    """Global correlation and nRMSE."""
    ft = y_true.reshape(-1)
    fp = y_pred.reshape(-1)
    m  = ~np.isnan(ft) & ~np.isnan(fp)
    ft, fp = ft[m], fp[m]
    if ft.size < 2:
        return 0.0, np.nan
    rmse      = np.sqrt(np.mean((fp - ft) ** 2))
    nrmse_pct = 100 * rmse / train_range if train_range > eps else np.nan
    corr      = (np.corrcoef(ft, fp)[0, 1]
                 if np.std(ft) > eps and np.std(fp) > eps else 0.0)
    return corr, nrmse_pct


def linear_regression_baseline_per_fold(
    data_dir, data_type, num_folds=5, alpha=1.0, output_dir=None, eps=1e-8
):
    """
    Ridge regression baseline evaluated per fold on BOTH train and test.
    Tests whether the CNN's complexity is justified over a linear model.
    """
    axis_labels    = ['X', 'Y', 'Z']
    all_fold_results = []

    print("\n" + "=" * 80)
    print("LINEAR REGRESSION BASELINE - PER-FOLD EVALUATION")
    print("=" * 80)
    print(f"Dataset : {data_dir}")
    print(f"Type    : {data_type}  |  Ridge alpha: {alpha}")
    print("=" * 80)

    for fold in range(num_folds):
        print(f"\n{'='*60}\nFOLD {fold}\n{'='*60}")
        fold_results = {"fold": fold, "axes": {}}

        train_npz   = join(data_dir, f"{fold}_fold_final_train.npz")
        test_npz    = join(data_dir, f"{fold}_fold_final_test.npz")
        scaler_path = join(data_dir, f"{fold}_fold_scaler4Y_{data_type}.pkl")

        if not os.path.exists(train_npz) or not os.path.exists(test_npz):
            print(f"  ❌ Missing NPZ files for fold {fold}. Skipping.")
            fold_results["error"] = "Missing NPZ files"
            all_fold_results.append(fold_results)
            continue

        try:
            train_data = np.load(train_npz)
            test_data  = np.load(test_npz)
            X_train = train_data["final_X_train"]
            X_test  = test_data["final_X_test"]
            if X_train.ndim == 3: X_train = X_train.reshape(X_train.shape[0], -1)
            if X_test.ndim  == 3: X_test  = X_test.reshape(X_test.shape[0],  -1)

            Y_tr = train_data[f"final_Y_{data_type}_train"]
            Y_te = test_data[f"final_Y_{data_type}_test"]

            with open(scaler_path, 'rb') as fh:
                scaler = pickle_load(fh)

            n_tr, n_te = Y_tr.shape[0], Y_te.shape[0]
            print(f"  Train: {n_tr}  Test: {n_te}  Features: {X_train.shape[1]}")

            Y_tr_r = Y_tr.reshape(n_tr, 3, 101)
            Y_te_r = Y_te.reshape(n_te, 3, 101)

            s_min   = scaler.min_.reshape(1, 3)
            s_scale = scaler.scale_.reshape(1, 3)

            Y_tr_u = np.stack([((Y_tr_r[i].T - s_min) / s_scale).T for i in range(n_tr)])
            Y_te_u = np.stack([((Y_te_r[i].T - s_min) / s_scale).T for i in range(n_te)])

        except Exception as e:
            print(f"  ❌ Error loading fold {fold}: {e}")
            fold_results["error"] = str(e)
            all_fold_results.append(fold_results)
            continue

        f_tr_corrs, f_tr_nrmses, f_te_corrs, f_te_nrmses = [], [], [], []

        for ax_idx, ax in enumerate(axis_labels):
            y_tr_ax = Y_tr_u[:, ax_idx, :]
            y_te_ax = Y_te_u[:, ax_idx, :]
            tr_range = max(np.nanmax(y_tr_ax) - np.nanmin(y_tr_ax), eps)

            tr_ok = ~np.isnan(y_tr_ax).any(axis=1) & ~np.isnan(X_train).any(axis=1)
            te_ok = ~np.isnan(y_te_ax).any(axis=1) & ~np.isnan(X_test).any(axis=1)

            mdl = Ridge(alpha=alpha, fit_intercept=True)
            mdl.fit(X_train[tr_ok], y_tr_ax[tr_ok])

            tr_c, tr_n = compute_metrics(y_tr_ax[tr_ok], mdl.predict(X_train[tr_ok]), tr_range)
            te_c, te_n = compute_metrics(y_te_ax[te_ok], mdl.predict(X_test[te_ok]),  tr_range)

            fold_results["axes"][ax] = {
                "train_corr": float(tr_c), "train_nrmse": float(tr_n),
                "test_corr":  float(te_c), "test_nrmse":  float(te_n),
                "n_train": int(tr_ok.sum()), "n_test": int(te_ok.sum()),
            }
            f_tr_corrs.append(tr_c); f_tr_nrmses.append(tr_n)
            f_te_corrs.append(te_c); f_te_nrmses.append(te_n)

        fold_results.update({
            "avg_train_corr":  float(np.mean(f_tr_corrs)),
            "avg_train_nrmse": float(np.mean(f_tr_nrmses)),
            "avg_test_corr":   float(np.mean(f_te_corrs)),
            "avg_test_nrmse":  float(np.mean(f_te_nrmses)),
            "avg_corr_gap":    float(np.mean(f_tr_corrs) - np.mean(f_te_corrs)),
            "avg_nrmse_gap":   float(np.mean(f_te_nrmses) - np.mean(f_tr_nrmses)),
        })
        all_fold_results.append(fold_results)

        print(f"  {'Axis':<6} {'Train Corr':>12} {'Train nRMSE':>12} {'Test Corr':>12} {'Test nRMSE':>12} {'Gap':>10}")
        print(f"  {'-'*70}")
        for ax in axis_labels:
            r = fold_results["axes"].get(ax, {})
            if "error" not in r:
                print(f"  {ax:<6} {r['train_corr']:>12.4f} {r['train_nrmse']:>11.2f}% "
                      f"{r['test_corr']:>12.4f} {r['test_nrmse']:>11.2f}% "
                      f"{r['train_corr']-r['test_corr']:>10.4f}")
        fr = fold_results
        print(f"  {'AVG':<6} {fr['avg_train_corr']:>12.4f} {fr['avg_train_nrmse']:>11.2f}% "
              f"{fr['avg_test_corr']:>12.4f} {fr['avg_test_nrmse']:>11.2f}% "
              f"{fr['avg_corr_gap']:>10.4f}")

    valid_folds = [f for f in all_fold_results if "avg_train_corr" in f]
    summary = {}
    if valid_folds:
        summary = {
            "num_folds":       len(valid_folds),
            "avg_train_corr":  float(np.mean([f["avg_train_corr"]  for f in valid_folds])),
            "std_train_corr":  float(np.std( [f["avg_train_corr"]  for f in valid_folds])),
            "avg_train_nrmse": float(np.mean([f["avg_train_nrmse"] for f in valid_folds])),
            "avg_test_corr":   float(np.mean([f["avg_test_corr"]   for f in valid_folds])),
            "std_test_corr":   float(np.std( [f["avg_test_corr"]   for f in valid_folds])),
            "avg_test_nrmse":  float(np.mean([f["avg_test_nrmse"]  for f in valid_folds])),
            "avg_corr_gap":    float(np.mean([f["avg_corr_gap"]    for f in valid_folds])),
            "avg_nrmse_gap":   float(np.mean([f["avg_nrmse_gap"]   for f in valid_folds])),
        }

        print("\n" + "=" * 80)
        print("LINEAR REGRESSION SUMMARY ACROSS ALL FOLDS")
        print("=" * 80)
        print(f"\n{'Metric':<25} {'Train':>22} {'Test':>22} {'Gap':>10}")
        print("-" * 82)
        print(f"{'Correlation':<25} {summary['avg_train_corr']:.4f} ± {summary['std_train_corr']:.4f}"
              f"   {summary['avg_test_corr']:.4f} ± {summary['std_test_corr']:.4f}"
              f"   {summary['avg_corr_gap']:.4f}")
        print(f"{'nRMSE (%)':<25} {summary['avg_train_nrmse']:.2f}%"
              f"   {summary['avg_test_nrmse']:.2f}%"
              f"   {summary['avg_nrmse_gap']:.2f}%")

        gap = summary['avg_corr_gap']
        label = ("⚠️  OVERFITTING DETECTED" if gap > 0.05
                 else "📊 MILD OVERFITTING"   if gap > 0.02
                 else "✅ GOOD GENERALIZATION")
        print(f"\n{label}: train-test gap = {gap:.4f}")
        print(f"Compare CNN test corr against linear baseline: {summary['avg_test_corr']:.3f}")

    out = {"baseline_type": "linear_regression", "alpha": alpha,
           "data_type": data_type, "folds": all_fold_results, "summary": summary}
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        p = join(output_dir, f"linear_regression_baseline_{data_type}.json")
        with open(p, 'w') as fh:
            json.dump(out, fh, indent=2)
        print(f"\n📁 Saved: {p}")

    return out


# --- Run linear regression baseline ---
linear_results = linear_regression_baseline_per_fold(
    data_dir=DATASET_DIR,
    data_type=DATA_TYPE,
    num_folds=5,
    alpha=1.0,
    output_dir=outputExcelBaseDir,
)

# --- Comparison table ---
print("\n" + "=" * 80)
print("BASELINE COMPARISON SUMMARY")
print("=" * 80)

naive_corr  = naive_results["summary"]["avg_corr"]      if naive_results and "summary" in naive_results else 0.0
naive_nrmse = naive_results["summary"]["avg_nrmse_pct"] if naive_results and "summary" in naive_results else float('nan')

if linear_results.get("summary"):
    s = linear_results["summary"]
    lr_tr_c, lr_tr_n = s["avg_train_corr"], s["avg_train_nrmse"]
    lr_te_c, lr_te_n = s["avg_test_corr"],  s["avg_test_nrmse"]
else:
    lr_tr_c = lr_tr_n = lr_te_c = lr_te_n = float('nan')

print(f"\n{'Baseline':<25} {'Train Corr':>12} {'Test Corr':>12} {'Train nRMSE':>12} {'Test nRMSE':>12}")
print("-" * 75)
print(f"{'Naive (Constant Mean)':<25} {'N/A':>12} {naive_corr:>12.3f} {'N/A':>12} {naive_nrmse:>11.2f}%")
print(f"{'Linear Regression':<25} {lr_tr_c:>12.4f} {lr_te_c:>12.4f} {lr_tr_n:>11.2f}% {lr_te_n:>11.2f}%")
print("-" * 75)
print(f"\nLinear vs Naive (Test): Corr +{lr_te_c - naive_corr:.3f},  nRMSE -{naive_nrmse - lr_te_n:.2f}pp")
